# Collaborative Filtering in Pure PyTorch

## What is collaborative filtering?

Collaborative filtering is a technique for predicting how much someone will like something, based on the collective behavior of many people. 
The core idea: if users who liked the same things as you also liked item X, you'll probably like item X too.

The term was coined by Goldberg et al. in 1992, originally for filtering email and news articles. 
Today it powers recommendations at Netflix, Spotify, Amazon, YouTube, and most platforms that show you personalized content.

What makes collaborative filtering distinctive is what it *doesn't* need: no descriptions, no genres, no metadata about the items. 
Just the pattern of who interacted with what. From that alone, the model discovers hidden structure.

### What kind of data?

All you need is an **interaction matrix** between two sets of entities. The classic case is users and items, but the technique is more general:

- **Users x Movies** - ratings (1-5 stars). What we'll build in this notebook.
- **Users x Products** - purchases, clicks. How Amazon recommends "customers who bought this also bought..."
- **Users x Songs** - play counts, skips. How Spotify builds Discover Weekly.
- **Words x Context words** - co-occurrence counts. Word2vec is mathematically a form of collaborative filtering.
- **Drugs x Proteins** - interaction data. Same technique, different domain entirely.

The interactions can be **explicit** (a 1-5 star rating) or **implicit** (a click, a purchase, a play). 
Explicit feedback is cleaner but rare - most real-world systems run on implicit signals because there's so much more of it.

### What problem does it solve?

Given a sparse matrix of interactions (most entries missing), **predict the missing entries**. 
If we can estimate how much User 42 would like a movie they haven't seen, we can rank all unseen movies and recommend the best ones.

This notebook builds a full collaborative filtering system from scratch in PyTorch. 
We'll start with intuition, build progressively better models, and end with a hybrid system that combines learned embeddings with real metadata - the approach used by most production recommendation systems.

In [ ]:
# Setup - run this first
from pathlib import Path
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

import urllib.request
import zipfile
import os
from pathlib import Path

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# === Plotting helpers (keeps logic cells clean) ===

def plot_losses(train_losses, valid_losses, title="Training Progress"):
    """Plot train/validation loss curves."""
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Train")
    plt.plot(valid_losses, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_crosstab(crosstab, title_to_id, title='How do we fill in the blanks?'):
    """Plot a user-movie rating heatmap with movie IDs bold, titles faded."""
    import pandas as pd_inner
    id_index = [f'{title_to_id.get(t, "?")}' for t in crosstab.index]
    original_titles = list(crosstab.index)
    crosstab = crosstab.copy()
    crosstab.index = id_index

    fig, ax = plt.subplots(figsize=(16, 10))
    sns.heatmap(crosstab, annot=True, fmt='.0f', cmap='YlOrRd',
                linewidths=0.5, linecolor='white',
                cbar_kws={'label': 'Rating'},
                mask=crosstab.isna(), ax=ax)
    sns.heatmap(crosstab.isna().astype(float), cmap=['#f5f5f5'], cbar=False,
                linewidths=0.5, linecolor='white', ax=ax, alpha=0.3)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('Movie')
    ax.set_xlabel('User ID')
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position('top')
    plt.xticks(rotation=0, fontsize=10, fontweight='bold')
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=10, fontweight='bold')
    for tick, t in zip(ax.get_yticks(), original_titles):
        ax.text(-0.5, tick, t, ha='right', va='center', fontsize=7, color='#999999')

    # Annotate the first empty cell with a big "?" and arrow
    for row_i in range(crosstab.shape[0]):
        found = False
        for col_i in range(crosstab.shape[1]):
            if pd.isna(crosstab.iloc[row_i, col_i]):
                # Place a big "?" in the center of the empty cell
                ax.text(col_i + 0.5, row_i + 0.5, '?', ha='center', va='center',
                        fontsize=22, fontweight='bold', color='#2563eb')
                # Arrow pointing at it from the right
                ax.annotate('', xy=(col_i + 0.8, row_i + 0.5),
                            xytext=(col_i + 2.5, row_i - 1.5),
                            arrowprops=dict(arrowstyle='->', color='#2563eb',
                                            lw=2.5, connectionstyle='arc3,rad=-0.2'))
                ax.text(col_i + 2.6, row_i - 1.7, 'Would they\nlike it?',
                        fontsize=11, fontweight='bold', color='#2563eb',
                        ha='left', va='bottom')
                found = True
                break
        if found:
            break

    plt.subplots_adjust(left=0.3)
    plt.show()

def plot_model_comparison(loss_data, titles):
    """Plot side-by-side loss curves for model comparison."""
    n = len(loss_data)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
    if n == 1: axes = [axes]
    for ax, (tl, vl), title in zip(axes, loss_data, titles):
        ax.plot(tl, label="Train")
        ax.plot(vl, label="Valid")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("MSE Loss")
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# === Diagram helpers ===

def plot_two_matrices(user_names, user_vals, movie_names, movie_vals):
    """Draw two embedding matrices side by side with dot product arrow."""
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 6)
    ax.axis('off')

    for side, names, vals, x_start, title, color, fc in [
        ('left', user_names, user_vals, 0.3, 'User Embeddings', '#1e40af', '#dbeafe'),
        ('right', movie_names, movie_vals, 6.8, 'Movie Embeddings', '#92400e', '#fef3c7')]:
        label_x = x_start
        cell_x = x_start + 1.2
        ax.text(cell_x + len(vals[0])*0.9/2 if vals[0] else cell_x, 5.7, title, ha='center', fontsize=12, fontweight='bold')
        for row, (name, v) in enumerate(zip(names, vals)):
            y = 5.0 - row * 0.9
            ax.text(label_x, y, name, ha='left', va='center', fontsize=10, fontweight='bold' if row==0 else 'normal')
            if v:
                for col, val in enumerate(v):
                    c = '#dbeafe' if side=='left' and row==0 else fc if side=='left' else '#fef3c7' if row==0 else '#f3f4f6'
                    rect = plt.Rectangle((cell_x + col*0.9, y-0.3), 0.8, 0.6, facecolor=c, edgecolor='#94a3b8', lw=1)
                    ax.add_patch(rect)
                    ax.text(cell_x + 0.4 + col*0.9, y, f'{val:+.1f}', ha='center', va='center', fontsize=9)
            else:
                ax.text(cell_x + 1, y, '...', ha='center', va='center', fontsize=12, color='#999')
    ax.annotate('', xy=(6.3, 1.2), xytext=(4.8, 1.2), arrowprops=dict(arrowstyle='->', color='#2563eb', lw=2))
    ax.text(5.55, 1.5, 'dot product', ha='center', fontsize=10, color='#2563eb', fontstyle='italic')
    plt.tight_layout()
    plt.show()

def plot_embedding_grid(matrix, row_names, highlight_row=None, title='Embedding matrix'):
    """Draw a matrix as a colored grid with one row optionally highlighted."""
    n_rows, n_cols = matrix.shape
    fig, ax = plt.subplots(figsize=(8, max(3, n_rows*0.8)))
    ax.axis('off')
    dims = [f'dim {j+1}' for j in range(n_cols)]
    cw, ch = 1.8, 0.7

    for j, d in enumerate(dims):
        ax.text(2.5 + j*cw + cw/2, n_rows*ch + 0.3, d, ha='center', fontsize=10, color='#888', fontstyle='italic')
    for i, (name, row) in enumerate(zip(row_names, matrix)):
        y = (n_rows - 1 - i) * ch
        hl = (i == highlight_row)
        ax.text(1.5, y + ch/2, name, ha='right', va='center', fontsize=11,
                fontweight='bold' if hl else 'normal', color='#2563eb' if hl else '#333')
        for j, v in enumerate(row):
            fc = '#dbeafe' if hl else '#f8fafc'
            ec = '#2563eb' if hl else '#cbd5e1'
            rect = plt.Rectangle((2.5 + j*cw, y), cw-0.05, ch-0.05, facecolor=fc, edgecolor=ec, lw=1.5 if hl else 0.8)
            ax.add_patch(rect)
            ax.text(2.5 + j*cw + cw/2, y + ch/2, f'{v:.2f}', ha='center', va='center', fontsize=11)
    if highlight_row is not None:
        hy = (n_rows - 1 - highlight_row) * ch
        ax.annotate(f'embedding[{highlight_row}]', xy=(2.5 + n_cols*cw + 0.3, hy + ch/2),
                    xytext=(2.5 + n_cols*cw + 1.5, hy + ch/2 - 0.6),
                    fontsize=11, fontweight='bold', color='#2563eb',
                    arrowprops=dict(arrowstyle='->', color='#2563eb', lw=2))
    ax.set_xlim(0, 2.5 + n_cols*cw + 3)
    ax.set_ylim(-0.8, n_rows*ch + 0.8)
    ax.set_title(title, fontsize=12, pad=15)
    plt.tight_layout()
    plt.show()

def plot_onehot_multiply(one_hot_vec, matrix, highlight_row):
    """Draw the one-hot x matrix = row diagram."""
    n_rows, n_cols = matrix.shape
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), gridspec_kw={'width_ratios': [1, 0.3, 3, 1.5]})
    for ax in axes: ax.axis('off')

    # One-hot
    ax = axes[0]
    ax.set_title('one-hot', fontsize=10)
    for i, v in enumerate(one_hot_vec):
        y = (n_rows - 1 - i) * 0.7
        fc = '#dbeafe' if v == 1 else '#f8fafc'
        ec = '#2563eb' if v == 1 else '#e2e8f0'
        rect = plt.Rectangle((0.2, y), 0.8, 0.6, facecolor=fc, edgecolor=ec, lw=2 if v==1 else 0.8)
        ax.add_patch(rect)
        ax.text(0.6, y+0.3, f'{int(v.item())}', ha='center', va='center', fontsize=12,
                fontweight='bold' if v==1 else 'normal', color='#2563eb' if v==1 else '#999')
    ax.set_xlim(-0.1, 1.3); ax.set_ylim(-0.2, n_rows*0.7)

    axes[1].text(0.5, n_rows*0.35, '@', ha='center', va='center', fontsize=16, color='#666')
    axes[1].set_xlim(0, 1); axes[1].set_ylim(-0.2, n_rows*0.7)

    # Matrix
    ax = axes[2]
    ax.set_title('embedding matrix', fontsize=10)
    for i in range(n_rows):
        y = (n_rows - 1 - i) * 0.7
        hl = (i == highlight_row)
        for j in range(n_cols):
            fc = '#dbeafe' if hl else '#f8fafc'
            ec = '#2563eb' if hl else '#e2e8f0'
            rect = plt.Rectangle((j*1.2, y), 1.1, 0.6, facecolor=fc, edgecolor=ec, lw=2 if hl else 0.8)
            ax.add_patch(rect)
            ax.text(j*1.2+0.55, y+0.3, f'{matrix[i,j].item():.2f}', ha='center', va='center', fontsize=10,
                    fontweight='bold' if hl else 'normal')
    ax.set_xlim(-0.2, n_cols*1.2 + 0.2); ax.set_ylim(-0.2, n_rows*0.7)

    # Result
    ax = axes[3]
    ax.text(0, n_rows*0.35, '=', ha='center', va='center', fontsize=16, color='#666')
    for j in range(n_cols):
        rect = plt.Rectangle((0.4, (n_cols-1-j)*0.7), 1.1, 0.6, facecolor='#dbeafe', edgecolor='#2563eb', lw=2)
        ax.add_patch(rect)
        ax.text(0.95, (n_cols-1-j)*0.7 + 0.3, f'{matrix[highlight_row, j].item():.2f}',
                ha='center', va='center', fontsize=10, fontweight='bold')
    ax.set_title(f'row {highlight_row}', fontsize=10, color='#2563eb', fontweight='bold')
    ax.set_xlim(-0.3, 1.8); ax.set_ylim(-0.2, n_rows*0.7)

    plt.suptitle('One-hot multiply picks out exactly one row', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

def plot_factor_comparison(results, factor_sizes):
    """Plot embedding size experiment results."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    for nf in factor_sizes:
        ax1.plot(results[nf]['valid'], label=f'{nf} factors')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Validation MSE')
    ax1.set_title('Validation loss by embedding size'); ax1.legend(); ax1.grid(True, alpha=0.3)
    bests = [results[nf]['best_valid'] for nf in factor_sizes]
    params = [results[nf]['params'] for nf in factor_sizes]
    colors = ['#3b82f6' if b == min(bests) else '#93c5fd' for b in bests]
    ax2.bar([f'{nf} factors\n({p:,} params)' for nf, p in zip(factor_sizes, params)], bests, color=colors)
    ax2.set_ylabel('Best Validation MSE'); ax2.set_title('Best validation loss vs model size'); ax2.grid(True, alpha=0.3, axis='y')
    plt.tight_layout(); plt.show()

def plot_dropout_comparison(dropout_results):
    """Plot dropout sweep results."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    for p, r in dropout_results.items():
        ax1.plot(r['valid'], label=f'p={p}')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Validation MSE')
    ax1.set_title('Validation loss by dropout rate'); ax1.legend(); ax1.grid(True, alpha=0.3)
    ps = list(dropout_results.keys())
    bests = [dropout_results[p]['best'] for p in ps]
    colors = ['#3b82f6' if b == min(bests) else '#93c5fd' for b in bests]
    ax2.bar([str(p) for p in ps], bests, color=colors)
    ax2.set_xlabel('Dropout rate'); ax2.set_ylabel('Best Validation MSE')
    ax2.set_title('Best validation loss vs dropout'); ax2.grid(True, alpha=0.3, axis='y')
    plt.tight_layout(); plt.show()

def plot_filled_crosstab(rows_data, top_users_list, top_movies_list, title_to_id):
    """Plot the filled-in crosstab: actual (blue) vs predicted (yellow)."""
    n_u, n_m = len(top_users_list), len(top_movies_list)
    fig, ax = plt.subplots(figsize=(16, 10))
    cw, ch = 1.0, 0.6
    for j, title in enumerate(top_movies_list):
        mid = title_to_id.get(title, '?')
        ax.text(2.5 + j*cw + cw/2, n_u*ch + 0.3, str(mid), ha='center', va='bottom', fontsize=8, fontweight='bold')
        ax.text(2.5 + j*cw + cw/2, n_u*ch + 0.55, title, ha='left', va='bottom', fontsize=6, rotation=45, color='#888')
    for i, uid in enumerate(top_users_list):
        y = (n_u - 1 - i) * ch
        ax.text(2.0, y + ch/2, str(uid), ha='right', va='center', fontsize=9, fontweight='bold')
        for j, title in enumerate(top_movies_list):
            val, kind = rows_data[i][title]
            x = 2.5 + j * cw
            if kind == 'actual':
                fc, ec, tc, fw = '#dbeafe', '#2563eb', '#1e40af', 'bold'
            else:
                fc, ec, tc, fw = '#fef3c7', '#d97706', '#92400e', 'normal'
            rect = plt.Rectangle((x, y), cw-0.04, ch-0.04, facecolor=fc, edgecolor=ec, lw=0.8)
            ax.add_patch(rect)
            ax.text(x + cw/2, y + ch/2, f'{val:.1f}', ha='center', va='center', fontsize=8, color=tc, fontweight=fw)
    ly = -0.8
    for color, ec, label, xo in [('#dbeafe','#2563eb','Actual rating',3), ('#fef3c7','#d97706','Predicted',7)]:
        rect = plt.Rectangle((xo, ly), 0.4, 0.35, facecolor=color, edgecolor=ec, lw=1)
        ax.add_patch(rect)
        ax.text(xo + 0.6, ly + 0.17, label, va='center', fontsize=9)
    ax.set_xlim(0, 2.5 + n_m*cw + 1); ax.set_ylim(ly - 0.3, n_u*ch + 2.5)
    ax.set_title('The blanks are filled in', fontsize=14, fontweight='bold')
    ax.axis('off'); plt.tight_layout(); plt.show()

def plot_vectors_crosstab(demo_users, demo_movies, user_vecs, movie_vecs):
    """Fastbook-style layout: user vectors left, movie vectors top, dot product grid center."""
    predictions = user_vecs @ movie_vecs.T
    n_u, n_m, n_f = len(demo_users), len(demo_movies), user_vecs.shape[1]
    cw, ch = 1.3, 0.65

    fig, ax = plt.subplots(figsize=(15, 8))
    ax.axis('off')

    user_vec_x = 1.5
    grid_x = user_vec_x + n_f * cw + 0.8
    top_y = n_u * ch + n_f * ch + 1.5
    movie_vec_y = top_y - n_f * ch
    grid_top_y = movie_vec_y - 1.2

    ax.set_xlim(-0.5, grid_x + n_m * cw + 1.5)
    ax.set_ylim(-0.5, top_y + 1.5)

    ax.text(grid_x + n_m*cw/2, top_y + 1.2,
            'Randomly initialized vectors - SGD will optimize these',
            ha='center', fontsize=11, fontstyle='italic', color='#666')

    for col, name in enumerate(demo_movies):
        ax.text(grid_x + col*cw + cw/2, top_y + 0.2, name,
                ha='left', va='bottom', fontsize=9, rotation=40, color='#92400e', fontweight='bold')

    ax.text(grid_x + n_m*cw/2, movie_vec_y + n_f*ch + 0.15,
            'Movie vectors', ha='center', fontsize=10, fontweight='bold', color='#d97706')
    for col, vec in enumerate(movie_vecs):
        for row, v in enumerate(vec):
            x = grid_x + col * cw
            y = movie_vec_y + (n_f - 1 - row) * ch
            rect = plt.Rectangle((x, y), cw-0.04, ch-0.04, facecolor='#fef3c7', edgecolor='#d97706', lw=0.8)
            ax.add_patch(rect)
            ax.text(x + cw/2, y + ch/2, f'{v:+.2f}', ha='center', va='center', fontsize=8, color='#92400e')

    for row in range(n_f):
        y = movie_vec_y + (n_f - 1 - row) * ch
        ax.text(grid_x + n_m*cw + 0.2, y + ch/2, f'dim {row+1}', ha='left', va='center', fontsize=8, color='#aaa')

    ax.text(user_vec_x + n_f*cw/2, grid_top_y + 0.2,
            'User vectors', ha='center', fontsize=10, fontweight='bold', color='#2563eb')
    for row, (name, vec) in enumerate(zip(demo_users, user_vecs)):
        y = grid_top_y - row * ch - ch
        ax.text(0, y + ch/2, name, ha='left', va='center', fontsize=10, fontweight='bold', color='#1e40af')
        for col, v in enumerate(vec):
            x = user_vec_x + col * cw
            rect = plt.Rectangle((x, y), cw-0.04, ch-0.04, facecolor='#dbeafe', edgecolor='#2563eb', lw=0.8)
            ax.add_patch(rect)
            ax.text(x + cw/2, y + ch/2, f'{v:+.2f}', ha='center', va='center', fontsize=8, color='#1e40af')

    ax.text(grid_x + n_m*cw/2, grid_top_y + 0.2,
            'Predicted ratings (dot products)', ha='center', fontsize=10, fontweight='bold', color='#555')
    for row in range(n_u):
        y = grid_top_y - row * ch - ch
        for col in range(n_m):
            x = grid_x + col * cw
            rect = plt.Rectangle((x, y), cw-0.04, ch-0.04, facecolor='#fafafa', edgecolor='#d1d5db', lw=0.8)
            ax.add_patch(rect)
            ax.text(x + cw/2, y + ch/2, f'{predictions[row, col]:.2f}', ha='center', va='center', fontsize=9)

    plt.tight_layout()
    plt.show()

def plot_pca_embeddings(coords, labels, n_show=50, title="Movie Embeddings (PCA -> 2D)", var_ratios=None):
    """Plot PCA-projected embeddings with movie title annotations."""
    plt.figure(figsize=(14, 14))
    plt.scatter(coords[:n_show, 0], coords[:n_show, 1], alpha=0.5)
    for i in range(n_show):
        plt.annotate(labels[i], (coords[i, 0], coords[i, 1]),
                     fontsize=9, color=np.random.RandomState(i).rand(3) * 0.7)
    plt.title(title)
    if var_ratios is not None:
        plt.xlabel(f"PC1 ({var_ratios[0]:.1%} variance)")
        plt.ylabel(f"PC2 ({var_ratios[1]:.1%} variance)")
    plt.grid(True, alpha=0.2)
    plt.show()

def plot_terminology():
    """Annotated embedding matrix showing terminology: matrix, embedding, latent factor."""
    names = ['User 0', 'User 1', 'User 2', 'User 3']
    vals = [[0.3, 0.8, -0.2, 0.5, 0.1],
            [-0.5, 0.1, 0.6, -0.3, 0.7],
            [0.7, -0.4, 0.2, 0.9, -0.1],
            [0.2, 0.6, -0.8, 0.4, 0.3]]
    cw, ch = 1.2, 0.65
    x0, y0 = 2.5, 0.3
    n_rows, n_cols = len(vals), len(vals[0])

    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.axis('off')

    for i in range(n_rows):
        ax.text(x0 - 0.3, y0 + (n_rows-1-i)*ch + ch/2, names[i], ha='right', va='center', fontsize=10)
        for j in range(n_cols):
            x = x0 + j*cw
            y = y0 + (n_rows-1-i)*ch
            fc = '#dbeafe' if i == 1 else '#f8fafc'
            ec = '#2563eb' if i == 1 else '#cbd5e1'
            lw = 2 if i == 1 else 0.8
            rect = plt.Rectangle((x, y), cw-0.04, ch-0.04, facecolor=fc, edgecolor=ec, lw=lw)
            ax.add_patch(rect)
            fw = 'bold' if (i == 1 and j == 2) else 'normal'
            fc2 = '#dc2626' if (i == 1 and j == 2) else '#333'
            ax.text(x + cw/2, y + ch/2, f'{vals[i][j]:.1f}', ha='center', va='center', fontsize=11, fontweight=fw, color=fc2)

    for j in range(n_cols):
        ax.text(x0 + j*cw + cw/2, y0 + n_rows*ch + 0.15, f'dim {j+1}', ha='center', fontsize=9, color='#888')

    ax.annotate('Embedding matrix\n(the whole table)',
                xy=(x0 - 0.5, y0 + n_rows*ch/2), xytext=(-0.5, y0 + n_rows*ch/2),
                fontsize=11, fontweight='bold', color='#333', va='center',
                arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

    row_y = y0 + (n_rows-1-1)*ch
    ax.annotate('One embedding\n(User 1\'s vector)',
                xy=(x0 + n_cols*cw + 0.2, row_y + ch/2),
                xytext=(x0 + n_cols*cw + 1.0, row_y + ch + 0.8),
                fontsize=11, fontweight='bold', color='#2563eb',
                arrowprops=dict(arrowstyle='->', color='#2563eb', lw=1.5))

    val_x = x0 + 2*cw + cw/2
    val_y = row_y + ch/2
    ax.annotate('One latent factor\n(a single learned number)',
                xy=(val_x, val_y - 0.35),
                xytext=(val_x + 1.5, y0 - 1.0),
                fontsize=11, fontweight='bold', color='#dc2626',
                arrowprops=dict(arrowstyle='->', color='#dc2626', lw=1.5))

    ax.set_xlim(-1.5, x0 + n_cols*cw + 4)
    ax.set_ylim(-1.5, y0 + n_rows*ch + 1)
    plt.tight_layout()
    plt.show()

def plot_weight_distributions(models_dict):
    """Compare embedding value distributions across models."""
    n = len(models_dict)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), sharey=True)
    if n == 1: axes = [axes]
    for ax, (name, model) in zip(axes, models_dict.items()):
        all_weights = []
        for param in model.parameters():
            all_weights.append(param.detach().cpu().flatten())
        all_weights = torch.cat(all_weights).numpy()
        ax.hist(all_weights, bins=80, color='steelblue', alpha=0.7, density=True)
        ax.axvline(0, color='red', linestyle='--', alpha=0.4)
        ax.set_title(name, fontsize=11)
        ax.set_xlabel('Parameter value')
        std = np.std(all_weights)
        mx = np.max(np.abs(all_weights))
        ax.text(0.95, 0.95, f'std={std:.3f}\nmax={mx:.2f}',
                transform=ax.transAxes, ha='right', va='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axes[0].set_ylabel('Density')
    plt.suptitle('Distribution of all parameter values', fontsize=12)
    plt.tight_layout()
    plt.show()


def plot_cosine_example(emb_a, emb_b, name_a, name_b, dims=None):
    """Compare two embedding vectors as bar charts to visualize similarity."""
    import numpy as _np
    n = len(emb_a)
    if dims is None:
        dims = [f'd{i+1}' for i in range(n)]
    cos = _np.dot(emb_a, emb_b) / (_np.linalg.norm(emb_a) * _np.linalg.norm(emb_b) + 1e-8)
    
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), gridspec_kw={'width_ratios': [3, 3, 2]})
    x = _np.arange(n)
    w = 0.35
    
    # Movie A bars
    axes[0].bar(x, emb_a, color='#3b82f6', alpha=0.8)
    axes[0].set_title(name_a, fontsize=11, fontweight='bold', color='#1e40af')
    axes[0].set_xticks(x); axes[0].set_xticklabels(dims, fontsize=8)
    axes[0].set_ylabel('Value'); axes[0].axhline(0, color='#ccc', lw=0.5)
    axes[0].set_ylim(-0.8, 0.8)
    
    # Movie B bars
    axes[1].bar(x, emb_b, color='#f59e0b', alpha=0.8)
    axes[1].set_title(name_b, fontsize=11, fontweight='bold', color='#92400e')
    axes[1].set_xticks(x); axes[1].set_xticklabels(dims, fontsize=8)
    axes[1].axhline(0, color='#ccc', lw=0.5)
    axes[1].set_ylim(-0.8, 0.8)
    
    # Side by side overlay
    axes[2].bar(x - w/2, emb_a, w, label=name_a, color='#3b82f6', alpha=0.7)
    axes[2].bar(x + w/2, emb_b, w, label=name_b, color='#f59e0b', alpha=0.7)
    axes[2].set_title(f'Cosine similarity: {cos:.2f}', fontsize=12, fontweight='bold',
                      color='#16a34a' if cos > 0.5 else '#dc2626' if cos < -0.5 else '#6b7280')
    axes[2].set_xticks(x); axes[2].set_xticklabels(dims, fontsize=8)
    axes[2].axhline(0, color='#ccc', lw=0.5)
    axes[2].legend(fontsize=8)
    axes[2].set_ylim(-0.8, 0.8)
    
    plt.tight_layout()
    plt.show()

def plot_collabnn_architecture():
    """Show how user + movie embeddings get concatenated into the MLP input."""
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axis('off')
    
    cw, ch = 0.7, 0.5
    
    # User embedding (blue, left)
    ax.text(1.5, 4.5, 'User embedding (50 dims)', fontsize=10, fontweight='bold', color='#1e40af', ha='center')
    for j in range(5):
        rect = plt.Rectangle((0.2 + j*cw, 3.7), cw-0.05, ch, facecolor='#dbeafe', edgecolor='#2563eb', lw=1)
        ax.add_patch(rect)
        ax.text(0.2 + j*cw + cw/2, 3.95, f'.{j+1}', ha='center', va='center', fontsize=8, color='#1e40af')
    ax.text(3.7, 3.95, '...', fontsize=12, color='#1e40af')
    
    # Movie embedding (orange, right)
    ax.text(6.5, 4.5, 'Movie embedding (50 dims)', fontsize=10, fontweight='bold', color='#92400e', ha='center')
    for j in range(5):
        rect = plt.Rectangle((4.7 + j*cw, 3.7), cw-0.05, ch, facecolor='#fef3c7', edgecolor='#d97706', lw=1)
        ax.add_patch(rect)
        ax.text(4.7 + j*cw + cw/2, 3.95, f'.{j+1}', ha='center', va='center', fontsize=8, color='#92400e')
    ax.text(8.2, 3.95, '...', fontsize=12, color='#92400e')
    
    # Concatenation arrow
    ax.annotate('concatenate', xy=(4.2, 2.9), xytext=(4.2, 3.5),
                fontsize=10, ha='center', color='#555', fontstyle='italic',
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
    
    # Combined vector (100 dims)
    ax.text(4.2, 2.7, 'Combined vector (100 values)', fontsize=10, fontweight='bold', ha='center')
    for j in range(5):
        rect = plt.Rectangle((0.2 + j*cw, 1.9), cw-0.05, ch, facecolor='#dbeafe', edgecolor='#2563eb', lw=0.8)
        ax.add_patch(rect)
    for j in range(5):
        rect = plt.Rectangle((4.7 + j*cw, 1.9), cw-0.05, ch, facecolor='#fef3c7', edgecolor='#d97706', lw=0.8)
        ax.add_patch(rect)
    ax.text(3.7, 2.15, '...', fontsize=12, color='#666')
    ax.text(8.2, 2.15, '...', fontsize=12, color='#666')
    
    # Arrow to MLP
    ax.annotate('', xy=(4.2, 1.3), xytext=(4.2, 1.8),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
    
    # MLP layers
    layers = [
        ('Linear(100 -> 100) + ReLU', '#e0e7ff'),
        ('Linear(100 -> 1)', '#e0e7ff'),
    ]
    for k, (label, fc) in enumerate(layers):
        y = 0.9 - k * 0.6
        rect = plt.Rectangle((1.5, y), 5.4, 0.45, facecolor=fc, edgecolor='#6366f1', lw=1, alpha=0.8)
        ax.add_patch(rect)
        ax.text(4.2, y + 0.22, label, ha='center', va='center', fontsize=10)
        if k < len(layers) - 1:
            ax.annotate('', xy=(4.2, y), xytext=(4.2, y - 0.1),
                        arrowprops=dict(arrowstyle='->', color='#555', lw=1))
    
    # Output
    ax.annotate('+ biases + sigmoid_range', xy=(4.2, -0.2), xytext=(4.2, 0.2),
                fontsize=10, ha='center', color='#555', fontstyle='italic',
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
    ax.text(4.2, -0.45, 'predicted rating', fontsize=11, fontweight='bold', color='#16a34a', ha='center')
    
    ax.set_xlim(-0.5, 9)
    ax.set_ylim(-0.8, 5)
    plt.tight_layout()
    plt.show()

def plot_hybrid_architecture():
    """Show how embeddings + metadata features get concatenated for the hybrid model."""
    fig, ax = plt.subplots(figsize=(13, 6))
    ax.axis('off')
    
    # Input boxes at the top
    inputs = [
        ('User\nembedding\n(50)', '#dbeafe', '#2563eb', 0, 1.8),
        ('Movie\nembedding\n(50)', '#fef3c7', '#d97706', 2.0, 1.8),
        ('Age\n(1)', '#f3f4f6', '#6b7280', 4.0, 0.8),
        ('Occupation\nembedding\n(8)', '#f3f4f6', '#6b7280', 5.0, 1.2),
        ('Genres\n(19)', '#f3f4f6', '#6b7280', 6.4, 1.0),
        ('Year\n(1)', '#f3f4f6', '#6b7280', 7.6, 0.8),
    ]
    
    top_y = 5.0
    for label, fc, ec, x, w in inputs:
        rect = plt.Rectangle((x, top_y), w, 1.2, facecolor=fc, edgecolor=ec, lw=1.5, alpha=0.9)
        ax.add_patch(rect)
        ax.text(x + w/2, top_y + 0.6, label, ha='center', va='center', fontsize=9, fontweight='bold')
    
    # Labels above
    ax.text(1.9, top_y + 1.5, 'Learned by SGD', ha='center', fontsize=9, fontstyle='italic', color='#2563eb')
    ax.text(6.2, top_y + 1.5, 'Known metadata', ha='center', fontsize=9, fontstyle='italic', color='#6b7280')
    
    # Arrows down to concatenation
    concat_y = 3.6
    for label, fc, ec, x, w in inputs:
        ax.annotate('', xy=(x + w/2, concat_y + 0.5), xytext=(x + w/2, top_y),
                    arrowprops=dict(arrowstyle='->', color='#aaa', lw=1))
    
    # Concatenation bar
    rect = plt.Rectangle((0, concat_y), 8.4, 0.45, facecolor='#e0e7ff', edgecolor='#6366f1', lw=1.5)
    ax.add_patch(rect)
    ax.text(4.2, concat_y + 0.22, 'Concatenate  [user_emb, movie_emb, age, occ_emb, genres, year]  =  129 values',
            ha='center', va='center', fontsize=10)
    
    # Arrow to MLP
    ax.annotate('', xy=(4.2, 2.6), xytext=(4.2, concat_y),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
    
    # MLP
    for k, label in enumerate(['Linear(129 -> 100) + ReLU', 'Linear(100 -> 1)']):
        y = 2.2 - k * 0.65
        rect = plt.Rectangle((1.5, y), 5.4, 0.5, facecolor='#e0e7ff', edgecolor='#6366f1', lw=1, alpha=0.8)
        ax.add_patch(rect)
        ax.text(4.2, y + 0.25, label, ha='center', va='center', fontsize=10)
    
    # Output
    ax.text(4.2, 0.7, '+ biases + sigmoid_range', ha='center', fontsize=10, color='#555', fontstyle='italic')
    ax.text(4.2, 0.2, 'predicted rating', ha='center', fontsize=12, fontweight='bold', color='#16a34a')
    
    ax.set_xlim(-0.5, 9)
    ax.set_ylim(-0.2, 7)
    plt.tight_layout()
    plt.show()


## Downloading the Data

We'll use [MovieLens 100K](https://grouplens.org/datasets/movielens/) - 100,000 ratings from 943 users on 1,682 movies. Small enough to train quickly, large enough to learn real patterns.

In [ ]:
DATA_DIR = Path("ml-100k")
if not DATA_DIR.exists():
    # Try relative path to repo data folder
    repo_data = Path("../../../data/ml-100k")
    if repo_data.exists():
        DATA_DIR = repo_data
    else:
        # Auto-download
        import urllib.request, zipfile
        urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-100k.zip", "/tmp/ml-100k.zip")
        with zipfile.ZipFile("/tmp/ml-100k.zip") as z:
            z.extractall(".")
        print("Downloaded MovieLens 100K")

print(f"Data directory: {DATA_DIR}")


## Part 1: The Data

Imagine you have a giant spreadsheet. Each row is one rating: "User 196 gave Movie 242 a score of 3." That's it. No descriptions, no genres - just who rated what and how much.

The main file is `u.data` - tab-separated with columns: user ID, movie ID, rating (1-5), timestamp. Let's load it and see what we're working with.

In [ ]:
# Load the ratings
ratings = pd.read_csv(
    DATA_DIR / "u.data",
    delimiter="\t",
    header=None,
    names=["user", "movie", "rating", "timestamp"]
)

# Load movie titles for interpretation later
movies = pd.read_csv(
    DATA_DIR / "u.item",
    delimiter="|",
    encoding="latin-1",
    usecols=(0, 1),
    names=("movie", "title"),
    header=None
)

# Merge so we have titles alongside ratings
ratings = ratings.merge(movies)
print(f"{len(ratings):,} ratings from {ratings['user'].nunique()} users on {ratings['title'].nunique()} movies")
ratings.head()

If you imagine this data as a big grid - users as rows, movies as columns - most cells would be empty. The average user has rated about 106 movies out of 1,664. That's a **sparse** matrix: over 93% empty.

Our job is to fill in those blanks. If we can predict what User 42 would rate a movie they haven't seen yet, we can recommend the ones we think they'd rate highest.

In [ ]:
# Visualize the sparse user-movie matrix
top_users = ratings.groupby('user')['rating'].count().nlargest(15).index
top_movies = ratings.groupby('title')['rating'].count().nlargest(15).index
title_to_id = dict(zip(movies['title'], movies['movie']))

subset = ratings[ratings['user'].isin(top_users) & ratings['title'].isin(top_movies)]
crosstab = subset.pivot_table(index='title', columns='user', values='rating')

plot_crosstab(crosstab, title_to_id)

total_possible = len(top_users) * len(top_movies)
filled = crosstab.notna().sum().sum()
print(f'Even among the most active users and popular movies: {filled}/{total_possible} cells filled ({filled/total_possible:.0%})')
n_u, n_m = ratings['user'].nunique(), ratings['title'].nunique()
print(f'Full dataset: {len(ratings)} ratings out of {n_u * n_m:,} possible ({len(ratings)/(n_u*n_m):.1%} filled)')


The white cells are missing ratings. 
For example, User 279 has rated many movies in our sample but hasn't rated Air Force One. 
Would they give it a 2 or a 5? That's the question.

Even among the most active users and popular movies, there are gaps. 
Across the full dataset, over 93% of the matrix is empty. 
Our model's job: fill in those blanks by discovering patterns in the ratings that do exist.

## Part 2: The Intuition - Latent Factors

Here's a thought experiment. Imagine you could describe every movie with a few numbers - 
how cerebral it is, how action-packed, how dark. 
And imagine every user had a matching set of numbers describing their taste - 
how much they value cerebral films, action, darkness.

If you had those numbers, predicting a rating would be trivial: 
just take the **dot product** (multiply pairs, sum - same operation from L3) and you'd get a match score. 
Let's pretend we have them and see how well this works.

In [ ]:
# Imagine we KNEW what qualities matter.
# 3 factors: [cerebral, action-packed, dark]

# The Matrix: very cerebral, very action-packed, quite dark
the_matrix = np.array([0.9, 0.95, 0.8])

# The Notebook: not cerebral, not action, not dark
the_notebook = np.array([-0.8, -0.9, -0.7])

# Alex: loves intense, brainy action
user_alex = np.array([0.85, 0.9, 0.7])


Each number represents how strongly that quality applies. 
Positive = yes, negative = no. 
The dot product multiplies each pair and sums - aligned tastes add up, mismatched tastes cancel out.

In [ ]:
# How well does Alex match The Matrix?
factors = ['cerebral', 'action-packed', 'dark']
products = user_alex * the_matrix

for name, u, m, p in zip(factors, user_alex, the_matrix, products):
    print(f'  {name:14s}  Alex={u:+.2f}  x  Matrix={m:+.2f}  =  {p:+.3f}')
print(f'{"":14s}  {"":28s}  sum = {products.sum():.2f}  <- high match!')


Every factor aligns: Alex likes cerebral, The Matrix is cerebral. Positive times positive = positive contribution. 
All three factors agree, so the sum is high.

What about a movie that's the opposite of what Alex likes?

In [ ]:
# Alex vs The Notebook - factors disagree
products2 = user_alex * the_notebook

for name, u, m, p in zip(factors, user_alex, the_notebook, products2):
    print(f'  {name:14s}  Alex={u:+.2f}  x  Notebook={m:+.2f}  =  {p:+.3f}')
print(f'{"":14s}  {"":32s}  sum = {products2.sum():.2f}  <- low match')


Positive times negative = negative contribution. 
Alex likes action (0.9), The Notebook isn't (-0.9) - that's a strong disagreement. 
The dot product goes negative.

But here's where it gets interesting. A different user with different tastes would get different scores from the *same* model:

In [ ]:
# Sam: loves romantic, lighthearted, non-action films
user_sam = np.array([-0.7, -0.5, -0.8])

print(f'Sam x The Notebook  = {(user_sam * the_notebook).sum():.2f}  <- high match!')
print(f'Sam x The Matrix    = {(user_sam * the_matrix).sum():.2f}  <- low match')


Same two movies, same factor vectors - but Sam gets the opposite recommendations from Alex. 
That's the power of this approach: one set of movie vectors serves every user. 
The user's vector decides what matters to *them*.

### The problem: metadata isn't enough

The demo above worked beautifully - but we made up the numbers. 
In reality, our dataset does have some metadata: movie genres, release year, user age, occupation. 
And those are useful - we'll use them later in the notebook.

But metadata is coarse. Genre tags can tell you The Matrix is "Action" and "Sci-Fi," but they can't capture 
what makes someone who loves The Matrix also love Amelie. 
There's no genre for "films with philosophical undertones and striking visual style." 
And there's no column in a user profile that says "enjoys unreliable narrators."

The richer the taste patterns, the harder they are to express as predefined features. 
You'd need an ever-growing vocabulary of categories, and someone would still need to label every movie and interview every user.

What if we could skip all that and let the data speak for itself?

### The solution: let gradient descent figure it out

Remember how neural networks work. In L3, we started with random weights and used gradient descent to find values that minimized prediction error. 
In L5, same thing with more layers. 
The weights started as meaningless noise, and after training they captured real patterns. 
Not because anyone told the model what to learn - but because SGD kept nudging them in the direction that made predictions less wrong.

We do the exact same thing here:

1. Give every user and every movie a vector of **random numbers**
2. Compute dot products to get predicted ratings
3. Compare predictions to actual ratings (the loss)
4. Use gradient descent to adjust the vectors
5. Repeat

That's it. The vectors are the model's parameters - just like weights in a neural network. SGD will shape them into something meaningful.

### Latent factors

After training, each movie's vector captures something real about that movie, and each user's vector captures something real about their taste. 
But nobody defined what the dimensions mean. 
Maybe dimension 12 ends up capturing "cerebral vs. popcorn." 
Maybe dimension 37 captures something humans wouldn't think to name.

These discovered dimensions are called **latent factors**: 
*latent* because they're hidden (discovered by training, not defined by a human), 
*factors* because they're the components that factor into the prediction.

### Embeddings

Those vectors of latent factors need to live somewhere. The model stores them in two matrices - one for users, one for movies. 
Here's what that looks like:

In [ ]:
user_names = ['Alex', 'Sam', 'User 72', '...']
user_vals = [[0.3, 0.8, -0.2, 0.5], [-.5, 0.1, 0.6, -.3], [0.7, -.4, 0.2, 0.9], None]
movie_names = ['The Matrix', 'The Notebook', 'Pulp Fiction', '...']
movie_vals = [[0.7, 0.9, 0.4, -0.1], [-.6, -.3, 0.8, 0.7], [0.5, 0.4, 0.6, 0.2], None]
plot_two_matrices(user_names, user_vals, movie_names, movie_vals)


Two lookup tables - one for users, one for movies. 
Every user gets one row, every movie gets one row. 
To predict a rating, grab one row from each and take their dot product.

The rows start as random numbers. Here's what that looks like when we try to use them:

In [ ]:
# Terminology: embedding matrix -> embeddings -> latent factors
plot_terminology()


**What is an embedding?** A learned vector representation - a list of numbers that started random and was shaped by training to capture patterns in the data. 
Similar items end up with similar vectors, because that's what minimized the loss. 
Unlike hand-crafted features (genre tags, age brackets), nobody defines what the numbers mean - SGD discovers whatever dimensions are useful.

**Terminology:**
- The **embedding matrix** is the full table (e.g. 943 users x 50 dimensions)
- One **embedding** is one row - a single user's or movie's complete vector
- Each number inside is one **latent factor** - a single hidden dimension that SGD discovered

When we say "50 latent factors," we mean each embedding has 50 numbers. 
When we say "look up the embedding for User 42," we mean grab row 42 from the user embedding matrix.

In [ ]:
# What does this look like concretely? Random vectors + dot product predictions
np.random.seed(3)
demo_users = ['Alex', 'Sam', 'User 72', 'User 405']
demo_movies = ['The Matrix', 'The Notebook', 'Pulp Fiction', 'Toy Story', 'Fargo']
user_vecs = np.random.randn(len(demo_users), 3) * 0.3
movie_vecs = np.random.randn(len(demo_movies), 3) * 0.3

plot_vectors_crosstab(demo_users, demo_movies, user_vecs, movie_vecs)


The predictions are nonsense - because the vectors are random. 
But this is exactly the same situation as L3: random weights, bad predictions, high loss. 
SGD will fix it, one gradient step at a time. 
After enough passes through the training data, these random numbers will become meaningful taste profiles and movie features - 
discovered entirely from the pattern of who rated what.

### How embeddings work

We have a matrix with one row per user. If the input is user 3, we want row 3. That row is the user's learned vector.

In [ ]:
# A small embedding matrix: 5 users, 3 factors
torch.manual_seed(0)
demo_matrix = torch.randn(5, 3) * 0.3
print('Embedding matrix:')
print(demo_matrix)

# User 3? Grab row 3.
print(f'\nUser 3 vector: {demo_matrix[3].tolist()}')


That's the practical mental model of an embedding: **a trainable table of vectors, indexed by integer**. 
Input a user ID, get back that user's vector. That's it.

### nn.Embedding

PyTorch provides `nn.Embedding` - a trainable lookup table. 
You give it an integer index, it returns the corresponding row, and SGD can update that row during training.

Mathematically, this is equivalent to multiplying a one-hot encoded vector by the matrix (which is why gradients work). 
But in practice, no one-hot vector is ever created - `nn.Embedding` just indexes directly into the matrix.

In [ ]:
# nn.Embedding: same result, no one-hot needed
emb = nn.Embedding(5, 3)

# Copy our demo values so we can compare
with torch.no_grad():
    emb.weight.copy_(demo_matrix)

# Look up user 3 - just pass the index
print(f'emb(3):          {emb(torch.tensor(3)).tolist()}')
print(f'demo_matrix[3]:  {demo_matrix[3].tolist()}')

# Look up multiple users at once
print(f'\nemb([0, 3, 4]):  batch lookup')
print(emb(torch.tensor([0, 3, 4])))


In practice, think: **embedding = trainable table of vectors**. 
You give it an integer index, it gives back the corresponding row, and SGD can update that row during training.

**Optional: why does this work with gradient descent?**

You might wonder: if an embedding is just "grab row 3," how can SGD compute gradients for that? 
Indexing doesn't sound like a differentiable operation.

The trick: "grab row 3" is mathematically identical to multiplying a one-hot vector `[0, 0, 0, 1, 0]` by the matrix. 
One-hot times matrix = the row where the 1 is. And matrix multiplication IS differentiable - so gradients flow back and update that row.

In practice, PyTorch never builds the one-hot vector. `nn.Embedding` indexes directly into the matrix but computes gradients as if it did the multiply. 
You don't need to think about this to use embeddings - just know that SGD can update them.

### Under the hood: nn.Parameter

What makes `nn.Embedding` different from a plain tensor? 
PyTorch only updates tensors that are wrapped in `nn.Parameter` - that's how the optimizer knows what to adjust:

In [ ]:
# Plain tensor: invisible to the optimizer
class ModelA(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = torch.ones(3)

# nn.Parameter: the optimizer will update this
class ModelB(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.ones(3))

# nn.Embedding: nn.Parameter + integer indexing
class ModelC(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Embedding(5, 3)

print(f'ModelA parameters: {list(ModelA().parameters())}')  # empty!
print(f'ModelB parameters: {[p.shape for p in ModelB().parameters()]}')
print(f'ModelC parameters: {[p.shape for p in ModelC().parameters()]}')


`nn.Embedding` = `nn.Parameter` + lookup by index. That's the full story.

Now we need to prepare our data for the model.

### Why "embeddings" and not just "weights"?

If you look at the PyTorch source code for `nn.Embedding`, it stores its data in a field called `.weight` - an `nn.Parameter` tensor, same as any linear layer. 
SGD updates it the same way. Gradients flow through it the same way. Mechanically, there's no difference.

So why the special name? The distinction is about what the numbers *represent*, not how they're computed:

- **Weights** in a linear layer define a *transformation* - they say "how much does input 3 contribute to output 7?" They connect layers.
- **Embeddings** *are* the representation of a specific entity - User 42's row doesn't transform anything, it IS User 42's learned profile.

The term comes from mathematics, where an embedding is a structure-preserving map from one space to another. 
In ML, it was popularized in NLP: Bengio et al. (2003) first learned dense word vectors inside a neural network, and Mikolov et al. (2013, word2vec) made the technique widely known. 
The idea was that you're "embedding" discrete items (words, users, movies) into a continuous vector space where similar items land near each other.

But under the hood? It's a parameter matrix and a row lookup. 
The fancy name describes the *concept* (learned representations of entities), not special math.

## Part 3: Preparing the Data

We understand the concept (latent factors), the data structure (embedding matrices), and the lookup mechanism (`nn.Embedding`). 
Now we need to prepare our actual MovieLens data for training.

### Mapping IDs to Indices

Embedding layers index into a matrix by row number: row 0, row 1, row 2... 
But our raw user and movie IDs have gaps and start from 1. 
We need to remap them to contiguous integers starting from 0.

In [ ]:
# Create contiguous ID mappings: original ID -> 0-based index
user_ids = ratings['user'].unique()
movie_ids = ratings['movie'].unique()

user2idx = {uid: idx for idx, uid in enumerate(user_ids)}
movie2idx = {mid: idx for idx, mid in enumerate(movie_ids)}

# Reverse mappings for interpretation later
idx2user = {idx: uid for uid, idx in user2idx.items()}
idx2movie = {idx: mid for mid, idx in movie2idx.items()}

# Movie index -> title (for printing recommendations)
movies_lookup = pd.read_csv(
    DATA_DIR / 'u.item', delimiter='|', encoding='latin-1',
    usecols=(0, 1), names=('movie', 'title'), header=None
)
movie_id2title = dict(zip(movies_lookup['movie'], movies_lookup['title']))
idx2title = {idx: movie_id2title[mid] for mid, idx in movie2idx.items()}

n_users = len(user2idx)
n_movies = len(movie2idx)

# Apply mappings
ratings['user_idx'] = ratings['user'].map(user2idx)
ratings['movie_idx'] = ratings['movie'].map(movie2idx)

# Show what the mapping does: original IDs -> clean indices
print('Before (original IDs - gaps, start from 1):')
sample = ratings[['user', 'movie', 'title']].head(5)
print(sample.to_string(index=False))

print(f'\nAfter (contiguous indices starting from 0):')
sample_mapped = ratings[['user', 'user_idx', 'movie', 'movie_idx', 'title']].head(5)
print(sample_mapped.to_string(index=False))

print(f'\n{n_users} users (indices 0-{n_users-1}), {n_movies} movies (indices 0-{n_movies-1})')
print(f'These indices map directly to rows in the embedding matrices.')


### Building the DataLoader

We need to feed (user_idx, movie_idx, rating) triplets to the model in batches. 
PyTorch splits this into two pieces:

- A **Dataset** holds all the data and knows how to return one item by index
- A **DataLoader** fetches many items, stacks them into batches, and shuffles

In [ ]:
class RatingsDataset(Dataset):
    """Each item is a (user_idx, movie_idx, rating) triplet."""
    def __init__(self, users, movies, ratings):
        # Store three parallel arrays as tensors
        self.users = torch.LongTensor(users)
        self.movies = torch.LongTensor(movies)
        self.ratings = torch.FloatTensor(ratings)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        # Returns one triplet: (user_idx, movie_idx, rating)
        return self.users[idx], self.movies[idx], self.ratings[idx]

# 80/20 train/validation split
train_df, valid_df = train_test_split(ratings, test_size=0.2, random_state=42)

# Create datasets - each takes three columns from the DataFrame
train_ds = RatingsDataset(
    train_df['user_idx'].values,   # all user indices
    train_df['movie_idx'].values,  # all movie indices
    train_df['rating'].values      # all ratings
)
valid_ds = RatingsDataset(valid_df['user_idx'].values, valid_df['movie_idx'].values, valid_df['rating'].values)



In [ ]:
# One item from the dataset: a single (user_idx, movie_idx, rating) triplet
u, m, r = train_ds[0]
print(f'One item:  user_idx={u}, movie_idx={m}, rating={r}')
print(f'           ({idx2title.get(m.item(), "?")})')

# The DataLoader batches many items together
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
valid_dl = DataLoader(valid_ds, batch_size=256)

users, movies, rats = next(iter(train_dl))
print(f'\nOne batch: 64 items stacked into tensors')
print(f'  users:   {users[:6].tolist()} ...  shape {users.shape}')
print(f'  movies:  {movies[:6].tolist()} ...  shape {movies.shape}')
print(f'  ratings: {rats[:6].tolist()} ...  shape {rats.shape}')

Each batch gives us 64 user indices, 64 movie indices, and 64 ratings. 
The model's job: given a user index and movie index, predict the rating. 
We have the intuition (dot products of latent factors), we have the mechanism (embeddings), and we have the data pipeline. Time to build the model.

## Part 4: The Dot Product Model

This is the simplest collaborative filtering model. Two embedding matrices (one for users, one for movies), and a dot product to predict ratings.

But there's a subtlety: the raw dot product can output any number - negative, larger than 5, anything. Ratings are 1-5. We need to squish the output into a valid range. That's what `sigmoid_range` does: it pushes the output through a sigmoid (which produces values between 0 and 1) and then stretches it to our desired range.

We use 0 to 5.5 instead of 1 to 5 because sigmoid *asymptotically approaches* its bounds but never quite reaches them. Using 5.5 gives the model room to actually predict 5.0.

In [ ]:
def sigmoid_range(x, low, high):
    """Squish any number into the range (low, high) using sigmoid."""
    return torch.sigmoid(x) * (high - low) + low

The raw dot product can output any number - negative, larger than 5, anything. 
But ratings are 1-5. We need to squish the output into a valid range.

A regular sigmoid maps any number to (0, 1). We scale it to (0, 5.5) instead. 
Why 5.5 and not 5? Because sigmoid asymptotically approaches its bounds but never reaches them - 
using 5.5 gives the model room to actually predict values close to 5.0.

In [ ]:
# What sigmoid_range does: squish any number into (0, 5.5)
test_inputs = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0, 5.0])
test_outputs = sigmoid_range(test_inputs, 0, 5.5)

print(f'{"Raw dot product":>15s}  ->  {"Predicted rating":>16s}')
print('-' * 38)
for inp, out in zip(test_inputs, test_outputs):
    print(f'{inp.item():>15.2f}  ->  {out.item():>16.2f} stars')


In [ ]:
class DotProduct(nn.Module):
    def __init__(self, n_users, n_movies, n_factors, y_range=(0, 5.5)):
        super().__init__()
        # Two embedding matrices - these ARE the model's learnable parameters
        self.user_factors = nn.Embedding(n_users, n_factors)   # (n_users, n_factors)
        self.movie_factors = nn.Embedding(n_movies, n_factors)  # (n_movies, n_factors)
        self.y_range = y_range

        # Initialize with small random values based on a normal distribution
        nn.init.normal_(self.user_factors.weight, 0, 0.1)
        nn.init.normal_(self.movie_factors.weight, 0, 0.1)

    def forward(self, users, movies):
        # Look up embedding vectors for each user and movie in the batch
        user_emb = self.user_factors(users)    # (batch, n_factors)
        movie_emb = self.movie_factors(movies)  # (batch, n_factors)
        # Element-wise multiply then sum = dot product per pair
        dot = (user_emb * movie_emb).sum(dim=1)
        # Clamp to valid rating range
        return sigmoid_range(dot, *self.y_range)

That's the **entire model**. Two embedding lookups, a dot product, and `sigmoid_range` to keep the output in bounds.

The only learnable parameters are the two embedding matrices - both start random. 
SGD's job is to adjust every number in both matrices so the dot products get closer to the actual ratings.

Now we need a training loop.

In [ ]:
def train_model(model, train_dl, valid_dl, n_epochs=5, lr=5e-3, wd=0.0):
    """Train a model and return loss history."""
    model = model.to(device)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=wd)  # wd = weight decay (L2 regularization)
    criterion = nn.MSELoss()  # mean squared error - how far off are our predictions?

    train_losses, valid_losses = [], []

    for epoch in range(n_epochs):
        # --- Training ---
        model.train()  # enable training mode (matters for dropout, batchnorm)
        batch_losses = []
        for users, mvs, rats in train_dl:  # each batch: 64 (user, movie, rating) triplets
            users, mvs, rats = users.to(device), mvs.to(device), rats.to(device)
            preds = model(users, mvs)       # forward: look up embeddings, dot product, sigmoid
            loss = criterion(preds, rats)    # how wrong were we?
            optimizer.zero_grad()            # clear previous gradients
            loss.backward()                  # compute new gradients
            optimizer.step()                 # nudge embedding values to reduce loss
            batch_losses.append(loss.item()) # track for reporting

        # --- Validation (no gradient updates, just measure performance) ---
        model.eval()  # disable dropout etc.
        all_preds, all_actual = [], []
        with torch.no_grad():  # don't compute gradients - saves memory and speed
            for users, mvs, rats in valid_dl:
                users, mvs, rats = users.to(device), mvs.to(device), rats.to(device)
                preds = model(users, mvs)
                all_preds.append(preds.cpu())
                all_actual.append(rats.cpu())

        all_preds = torch.cat(all_preds)    # combine all batches into one tensor
        all_actual = torch.cat(all_actual)
        train_loss = np.mean(batch_losses)
        valid_loss = ((all_preds - all_actual) ** 2).mean().item()  # MSE on full validation set
        rmse = valid_loss ** 0.5  # root MSE - in units of stars
        within_1 = ((all_preds - all_actual).abs() <= 1.0).float().mean().item()  # % predictions within 1 star
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)
        print(f"Epoch {epoch+1:2d}/{n_epochs}  MSE={train_loss:.4f}  val_MSE={valid_loss:.4f}  RMSE={rmse:.2f}  within_1_star={within_1:.0%}")

    return train_losses, valid_losses


The training loop follows the exact same pattern we used for regression in L3 and neural networks in L5:

1. **Forward pass**: feed data through the model to get predictions
2. **Loss**: compute how wrong the predictions are (MSE - mean squared error)
3. **Backward pass**: compute gradients (how should each weight change to reduce loss?)
4. **Update**: nudge each weight in the direction that reduces loss

The only difference from a neural network is *what* the model looks like inside. Instead of linear layers and activations, we have embedding lookups and a dot product. But SGD doesn't care - it just follows gradients.

We also track validation loss separately. Validation data is never used for training - it tells us how well the model generalizes to ratings it hasn't memorized.

In [ ]:
# Train the dot product model with 50 latent factors
n_factors = 50
model_v1 = DotProduct(n_users, n_movies, n_factors)

print('Training metrics per epoch:')
print('  MSE       = training loss (mean squared error - lower is better)')
print('  val_MSE   = validation loss (how well we predict unseen ratings)')
print('  RMSE      = root MSE in stars (e.g. 1.07 = off by ~1 star on average)')
print('  within_1  = % of predictions within 1 star of the actual rating')
print()

train_losses_v1, valid_losses_v1 = train_model(model_v1, train_dl, valid_dl, n_epochs=15, lr=1e-2)


**Reading the metrics:** RMSE tells you how far off predictions are on average, in stars. 
An RMSE of 1.07 means "typically off by about 1 star." 
The `within_1_star` metric is more intuitive - 67% means two thirds of predictions land within 1 star of the actual rating.

But look at the validation loss - it's best at epoch 1 and gets worse from there. 
The model overshoots immediately and then memorizes training data. 
This is a sign that our learning rate (`lr=1e-2`) is too aggressive.

In [ ]:
plot_losses(train_losses_v1, valid_losses_v1, 'DotProduct (lr=1e-2) - overfitting from epoch 1')


### Fixing the learning rate

Before adding anything else, let's try a lower learning rate. 
At `lr=1e-2`, the model takes steps that are too large - it overshoots the good region immediately. 
At `lr=1e-3`, it takes smaller steps and has time to actually learn.

In [ ]:
# Same model, lower learning rate
model_v1b = DotProduct(n_users, n_movies, n_factors)
train_losses_v1b, valid_losses_v1b = train_model(model_v1b, train_dl, valid_dl, n_epochs=15, lr=1e-3)


In [ ]:
plot_losses(train_losses_v1b, valid_losses_v1b, 'DotProduct (lr=1e-3) - actual training')


Now the model actually improves over multiple epochs before overfitting sets in. 
The best validation RMSE is also better. 
Learning rate is one of the most important hyperparameters - always try a few values.

We'll use `lr=1e-3` from here on.

### How many latent factors?

The embedding dimension controls how much the model can express:
- **Too few** (e.g. 5): the model can't capture enough nuance.
- **Too many** (e.g. 200): more parameters than the data can support.

There's no formula with solid theory behind it. 
fastai uses `min(600, round(1.6 * n^0.56))` - an empirical power law that Jeremy Howard curve-fit against what worked on real datasets. 
Google's TensorFlow docs suggest `n^0.25`, which is much more conservative. 
In practice, you try a few values and compare.

In [ ]:
# Experiment: how does embedding size affect performance?
factor_sizes = [5, 50, 200]
results = {}

for nf in factor_sizes:
    print(f'--- Training with {nf} factors ---')
    model = DotProduct(n_users, n_movies, nf)
    tl, vl = train_model(model, train_dl, valid_dl, n_epochs=15, lr=1e-3)
    results[nf] = {'train': tl, 'valid': vl, 'best_valid': min(vl),
                   'params': sum(p.numel() for p in model.parameters())}
    print(f'  Best valid MSE: {min(vl):.4f}  |  Parameters: {results[nf]["params"]:,}\n')


In [ ]:
# Compare the three runs
plot_factor_comparison(results, factor_sizes)


With a proper learning rate, all three sizes actually train. 
50 factors is a reasonable default for this dataset - it has enough capacity without extreme overfitting. 
We'll stick with 50.

## Part 5: Adding Biases

Think about it: some users are generous raters - they give everything 4-5 stars. Others are harsh critics - their 3 stars is actually high praise. 
And some movies are just universally good: even if your taste doesn't particularly align with Shawshank Redemption, you'd still rate it higher than average.

The dot product can only capture taste *match* - it multiplies user preferences with movie qualities. 
It has no way to say "this user tends to rate everything high" or "everyone likes this movie." Those are baseline offsets, not taste interactions.

Fix: add a learnable **bias** per user and per movie.

```
prediction = dot(user_vec, movie_vec) + user_bias + movie_bias
```

In [ ]:
class DotProductBias(nn.Module):
    def __init__(self, n_users, n_movies, n_factors, y_range=(0, 5.5)):
        super().__init__()
        self.user_factors = nn.Embedding(n_users, n_factors)    # taste vectors
        self.user_bias = nn.Embedding(n_users, 1)               # one bias per user
        self.movie_factors = nn.Embedding(n_movies, n_factors)   # quality vectors
        self.movie_bias = nn.Embedding(n_movies, 1)              # one bias per movie
        self.y_range = y_range

        # Initialize: factors with small random values, biases at zero
        for emb in [self.user_factors, self.movie_factors]:
            nn.init.normal_(emb.weight, 0, 0.1)
        for emb in [self.user_bias, self.movie_bias]:
            nn.init.zeros_(emb.weight)

    def forward(self, users, movies):
        user_emb = self.user_factors(users)       # (batch, n_factors)
        movie_emb = self.movie_factors(movies)     # (batch, n_factors)
        # Dot product: how well does taste match?
        dot = (user_emb * movie_emb).sum(dim=1)
        # Add biases: shift based on user generosity + movie quality
        dot += self.user_bias(users).squeeze() + self.movie_bias(movies).squeeze()
        return sigmoid_range(dot, *self.y_range)

Notice the biases are `nn.Embedding(n, 1)` - one number per user/movie, looked up the same way as the factor vectors. 
We initialize them at zero (no bias assumed) and let SGD learn the right values.

Does adding biases actually help? Let's train and compare.

In [ ]:
# Train with biases, same lr=1e-3
model_v2 = DotProductBias(n_users, n_movies, n_factors)
train_losses_v2, valid_losses_v2 = train_model(model_v2, train_dl, valid_dl, n_epochs=15, lr=1e-3)


### Weight Decay

Biases give the model more expressiveness, but also more room to overfit. 
**Weight decay** (also called L2 regularization) adds a penalty to the loss that punishes large parameter values:

```
total_loss = MSE_loss + wd * sum(all_weights^2)
```

Two forces now act on each parameter:
- **Prediction loss** pulls the parameter toward whatever value best predicts ratings
- **Weight decay** pulls the parameter toward zero

A parameter only stays large if it genuinely helps predictions enough to justify the penalty. 
Think of it like gravity - useful structures stay up, flimsy ones collapse.

Let's train with weight decay and see what it does to the actual embedding values.

In [ ]:
# Train with weight decay (wd=1e-4 works well at this learning rate)
model_v3 = DotProductBias(n_users, n_movies, n_factors)
train_losses_v3, valid_losses_v3 = train_model(model_v3, train_dl, valid_dl, n_epochs=15, lr=1e-3, wd=1e-4)


In [ ]:
# What does weight decay actually do to the embeddings?
plot_weight_distributions({
    'No weight decay (model_v2)': model_v2,
    'With weight decay (model_v3)': model_v3,
})


Without weight decay, the parameter values spread out - some get large to memorize specific training examples. 
With weight decay, the distribution is tighter around zero. 
The model is forced to use smaller, more conservative values that capture general patterns instead of noise.

In [ ]:
# Compare all models so far
import math
print(f'{"Model":<30s}  {"Best Valid MSE":>14s}  {"RMSE (stars)":>12s}')
print('-' * 60)
for name, vl in [
    ('DotProduct (lr=1e-2)', valid_losses_v1),
    ('DotProduct (lr=1e-3)', valid_losses_v1b),
    ('DotProductBias', valid_losses_v2),
    ('DotProductBias + wd', valid_losses_v3)]:
    best = min(vl)
    print(f'{name:<30s}  {best:>14.4f}  {math.sqrt(best):>12.2f}')

plot_model_comparison(
    [(train_losses_v1b, valid_losses_v1b), (train_losses_v2, valid_losses_v2), (train_losses_v3, valid_losses_v3)],
    ['DotProduct', 'DotProductBias', 'DotProductBias + wd']
)


Each step helps: lowering the learning rate let the model actually train. 
Biases capture baseline tendencies (generous raters, universally liked movies). 
Weight decay prevents the extra parameters from overfitting.

### More regularization: early stopping and dropout

Weight decay helped, but what else can we try? 
Two more common techniques:

**Early stopping** saves the best model and stops when validation loss hasn't improved for `patience` epochs. 
Instead of using the final (overfit) model, we restore the best one.

In [ ]:
# Early stopping: save the best model, stop when val loss stops improving
def train_model_es(model, train_dl, valid_dl, n_epochs=30, lr=1e-3, wd=1e-4, patience=5):
    """Train with early stopping. Returns loss history."""
    model = model.to(device)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.MSELoss()
    
    train_losses, valid_losses = [], []
    best_val_loss = float('inf')
    best_state = None
    wait = 0
    
    for epoch in range(n_epochs):
        model.train()
        batch_losses = []
        for users, mvs, rats in train_dl:
            users, mvs, rats = users.to(device), mvs.to(device), rats.to(device)
            preds = model(users, mvs)
            loss = criterion(preds, rats)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
        
        model.eval()
        all_preds, all_actual = [], []
        with torch.no_grad():
            for users, mvs, rats in valid_dl:
                users, mvs, rats = users.to(device), mvs.to(device), rats.to(device)
                all_preds.append(model(users, mvs).cpu())
                all_actual.append(rats.cpu())
        
        all_preds = torch.cat(all_preds)
        all_actual = torch.cat(all_actual)
        tl = np.mean(batch_losses)
        vl = ((all_preds - all_actual) ** 2).mean().item()
        rmse = ((all_preds - all_actual) ** 2).mean().sqrt().item()
        within_1 = ((all_preds - all_actual).abs() <= 1.0).float().mean().item()
        train_losses.append(tl)
        valid_losses.append(vl)
        
        marker = ''
        if vl < best_val_loss:
            best_val_loss = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
            marker = ' *'
        else:
            wait += 1
        
        print(f'Epoch {epoch+1:2d}/{n_epochs}  MSE={tl:.4f}  val_MSE={vl:.4f}  RMSE={rmse:.2f}  within_1={within_1:.0%}{marker}')
        
        if wait >= patience:
            print(f'Early stopping at epoch {epoch+1} (best was epoch {epoch+1-patience})')
            break
    
    model.load_state_dict(best_state)
    return train_losses, valid_losses

# Train with early stopping + AdamW
model_es = DotProductBias(n_users, n_movies, n_factors)
tl_es, vl_es = train_model_es(model_es, train_dl, valid_dl, n_epochs=30, lr=1e-3, wd=1e-4, patience=5)


**Dropout on embeddings** randomly zeros out parts of the embedding vector during training, 
forcing the model to not rely on any single latent factor too heavily. 
More dropout = the model trains longer before overfitting, because it can't memorize through any single dimension.

In [ ]:
# DotProductBias with dropout on embeddings
class DotProductBiasDropout(nn.Module):
    def __init__(self, n_users, n_movies, n_factors, y_range=(0, 5.5), p=0.1):
        super().__init__()
        self.user_factors = nn.Embedding(n_users, n_factors)
        self.user_bias = nn.Embedding(n_users, 1)
        self.movie_factors = nn.Embedding(n_movies, n_factors)
        self.movie_bias = nn.Embedding(n_movies, 1)
        self.drop = nn.Dropout(p)
        self.y_range = y_range

        for emb in [self.user_factors, self.movie_factors]:
            nn.init.normal_(emb.weight, 0, 0.1)
        for emb in [self.user_bias, self.movie_bias]:
            nn.init.zeros_(emb.weight)

    def forward(self, users, movies):
        user_emb = self.drop(self.user_factors(users))
        movie_emb = self.drop(self.movie_factors(movies))
        dot = (user_emb * movie_emb).sum(dim=1)
        dot += self.user_bias(users).squeeze() + self.movie_bias(movies).squeeze()
        return sigmoid_range(dot, *self.y_range)

# Try different dropout rates
dropout_results = {}
for p in [0.0, 0.1, 0.2, 0.3, 0.5]:
    print(f'--- Dropout p={p} ---')
    m = DotProductBiasDropout(n_users, n_movies, n_factors, p=p)
    tl, vl = train_model_es(m, train_dl, valid_dl, n_epochs=30, lr=1e-3, wd=1e-4, patience=5)
    dropout_results[p] = {'train': tl, 'valid': vl, 'best': min(vl)}
    print(f'  Best valid MSE: {min(vl):.4f}\n')


In [ ]:
# Compare dropout rates
plot_dropout_comparison(dropout_results)


### What worked?

Let's put all our experiments side by side.

In [ ]:
# Final comparison table
import math

all_results = [
    ('DotProduct (lr=1e-2)', min(valid_losses_v1)),
    ('DotProduct (lr=1e-3)', min(valid_losses_v1b)),
    ('DotProductBias', min(valid_losses_v2)),
    ('DotProductBias + wd', min(valid_losses_v3)),
    ('Early stopping (AdamW)', min(vl_es)),
]

# Add best dropout result
best_p = min(dropout_results, key=lambda p: dropout_results[p]['best'])
all_results.append((f'Dropout (p={best_p})', dropout_results[best_p]['best']))

print(f'{"Model":<30s}  {"Best Valid MSE":>14s}  {"RMSE":>6s}')
print('-' * 55)
for name, mse in all_results:
    print(f'{name:<30s}  {mse:>14.4f}  {math.sqrt(mse):>6.2f}')


The biggest single improvement was fixing the learning rate (1e-2 to 1e-3). 
After that, biases and weight decay each helped incrementally. 
The dropout and early stopping experiments show that regularization matters, 
though the exact ranking of techniques varies with random initialization.

For the interpretation sections that follow, we'll use model_v3 (DotProductBias with weight decay) since it's the simplest regularized model and the easiest to extract biases and embeddings from.

## Part 6: What Can We Do With This?

We trained a model on ratings. What did that actually give us? Four useful assets:

- **A user embedding** - a compressed representation of each user's taste (50 numbers)
- **A movie embedding** - a compressed representation of what kind of users like each movie (50 numbers)
- **User and movie biases** - how generous a rater is, how broadly liked a movie is
- **A scoring function** - given any user and movie, estimate how much they'd like it

The predicted rating itself ("3.8 stars") is less important than what it enables: **ranking**. 
Users don't care whether the system predicts 4.1 vs 4.3. 
They care whether the top 10 items shown to them are compelling. 
Let's see the practical use cases.

### Top picks for a user

The simplest use: for a given user, score all movies they haven't seen yet, and show the highest-scoring ones.

In [ ]:
# Pick a user and predict their rating for every movie
model_v3.eval()
user_embs = model_v3.user_factors.weight.detach().cpu()
movie_embs = model_v3.movie_factors.weight.detach().cpu()
user_biases = model_v3.user_bias.weight.detach().cpu().squeeze()
movie_biases = model_v3.movie_bias.weight.detach().cpu().squeeze()

def predict_rating(user_idx, movie_idx):
    """Predict one rating using the trained model."""
    dot = (user_embs[user_idx] * movie_embs[movie_idx]).sum()
    score = dot + user_biases[user_idx] + movie_biases[movie_idx]
    return sigmoid_range(score, 0, 5.5).item()

def predict_all_for_user(user_idx):
    """Predict ratings for all movies for one user."""
    dots = (movie_embs * user_embs[user_idx]).sum(dim=1)
    scores = dots + user_biases[user_idx] + movie_biases
    return sigmoid_range(scores, 0, 5.5)

# Predict for User 0, but only show movies they haven't rated
preds = predict_all_for_user(0)
already_rated = set(ratings[ratings['user_idx'] == 0]['movie_idx'].values)
unseen_preds = [(idx, preds[idx].item()) for idx in range(len(preds)) if idx not in already_rated]
unseen_preds.sort(key=lambda x: x[1], reverse=True)

print(f'User 0 has rated {len(already_rated)} movies. Top 10 recommendations from the {len(unseen_preds)} unseen:')
print(f'{"Rank":<6s} {"Movie":<45s} {"Predicted":>9s}')
print('-' * 62)
for rank, (idx, score) in enumerate(unseen_preds[:10], 1):
    print(f'{rank:<6d} {idx2title[idx]:<45s} {score:.1f} stars')


In [ ]:
# What about movies this user has already rated? How close are our predictions?
user_0_ratings = ratings[ratings['user_idx'] == 0][['title', 'rating', 'movie_idx']]
user_0_ratings = user_0_ratings.copy()
user_0_ratings['predicted'] = user_0_ratings['movie_idx'].apply(
    lambda m: predict_rating(0, m)
)
user_0_ratings['error'] = (user_0_ratings['rating'] - user_0_ratings['predicted']).abs()

print(f'User 0 rated {len(user_0_ratings)} movies. Sample predictions vs actual:')
print()
sample = user_0_ratings.sort_values('rating', ascending=False).head(10)
print(f'{"Movie":<45s} {"Actual":>6s} {"Predicted":>9s} {"Error":>6s}')
print('-' * 70)
for _, row in sample.iterrows():
    print(f'{row["title"]:<45s} {row["rating"]:>6.0f} {row["predicted"]:>9.1f} {row["error"]:>6.1f}')
print(f'\nAverage error across all {len(user_0_ratings)} rated movies: {user_0_ratings["error"].mean():.2f} stars')


The model isn't perfect - it's off by about a star on some movies - but the predictions generally track the actual ratings. 
The exact number matters less than the *ordering* - and the ordering is what drives recommendations.

Now let's go back to our original question from Part 1: can we fill in the blanks?

### Filling in the blanks

Remember our crosstab from Part 1 with all the missing ratings? Let's fill them in.

In [ ]:
# Fill in the blanks from our original crosstab
# Reload movie titles (the 'movies' variable was shadowed by training loop variables)
movies_df = pd.read_csv(
    DATA_DIR / "u.item", delimiter="|", encoding="latin-1",
    usecols=(0, 1), names=("movie", "title"), header=None
)
top_users_list = ratings.groupby('user')['rating'].count().nlargest(15).index.tolist()
top_movies_list = ratings.groupby('title')['rating'].count().nlargest(15).index.tolist()

# Build a filled crosstab: actual ratings where we have them, predictions where we don't
rows = []
for uid in top_users_list:
    uidx = user2idx[uid]
    row = {}
    for title in top_movies_list:
        mid = movies_df[movies_df['title'] == title]['movie'].values[0]
        midx = movie2idx[mid]
        # Check if this user actually rated this movie
        actual = ratings[(ratings['user'] == uid) & (ratings['title'] == title)]['rating'].values
        if len(actual) > 0:
            row[title] = (actual[0], 'actual')
        else:
            row[title] = (predict_rating(uidx, midx), 'predicted')
    rows.append(row)

# Visualize: actual ratings in dark, predictions in light

plot_filled_crosstab(rows, top_users_list, top_movies_list, title_to_id)


Blue cells are actual ratings. Yellow cells are predictions - the blanks we set out to fill. 
This is the core of collaborative filtering: turn a sparse matrix of interactions into a dense matrix of predictions. Sometimes people say "matrix completion" or "matrix factorization", that's what we've done here.

### Similar items: "Because you watched..."

The second major use case doesn't need a user at all. 
Given one movie, find others with similar embedding vectors.

Each movie's embedding is a row of 50 numbers. 
To find similar movies, we compare the *pattern* of numbers across two rows. 
**Cosine similarity** measures whether two vectors have the same shape - the same pattern of highs and lows - regardless of scale:

- **cos near 1**: same pattern (similar movies)
- **cos near 0**: unrelated patterns
- **cos near -1**: opposite pattern

In [ ]:
# Compare embedding patterns for two similar movies and two different ones
# Using first 8 dims for readability
embs = model_v3.movie_factors.weight.detach().cpu().numpy()

# Find Star Wars and Empire Strikes Back indices
sw_idx = [i for i, t in idx2title.items() if 'Star Wars' in t][0]
esb_idx = [i for i, t in idx2title.items() if 'Empire Strikes' in t][0]
sc_idx = [i for i, t in idx2title.items() if 'Schindler' in t][0]

print('Similar movies - same pattern of highs and lows:')
plot_cosine_example(embs[sw_idx][:8], embs[esb_idx][:8],
                    'Star Wars', 'Empire Strikes Back',
                    dims=[f'd{i+1}' for i in range(8)])

print('Different movies - different patterns:')
plot_cosine_example(embs[sw_idx][:8], embs[sc_idx][:8],
                    'Star Wars', "Schindler's List",
                    dims=[f'd{i+1}' for i in range(8)])


When two movies have the same pattern - bars go up and down in the same places - cosine similarity is high. 
When the patterns are different, it's low. 
The actual magnitude of the bars doesn't matter, only the shape.

Why not just compare the raw numbers (Euclidean distance)? 
Because popular movies tend to have larger embedding values (more gradient updates pushed them further from zero). 
Cosine ignores the scale and focuses on the pattern, which is what actually represents taste.
However, you could actually use  both, e.g if you use weight decay Euclidean might work well and having a large enough dataset such that embeddings actually end up getting updated properly. 

Let's use this to build a "because you watched" recommendation engine:

<details>
<summary><strong>Optional deep dive: how cosine similarity actually works</strong></summary>

You can skip this - the intuition above ("same pattern of highs and lows") is enough to use cosine similarity. 
This section shows the actual math for the curious.

The question: given two rows of numbers, how do we measure whether they follow the same pattern?
</details>

In [ ]:
# How cosine similarity works - step by step with real numbers
import numpy as _np

a = _np.array([0.5, 0.3, -0.2])
b = _np.array([0.4, 0.2, -0.1])   # similar to A
c = _np.array([1.0, 0.6, -0.4])   # A doubled - same pattern, bigger numbers

print('='*60)
print('THREE MOVIE EMBEDDING VECTORS (3 dims for simplicity)')
print('='*60)
print(f'  Movie A: {a.tolist()}')
print(f'  Movie B: {b.tolist()}   <- similar pattern, similar scale')
print(f'  Movie C: {c.tolist()}   <- same pattern as A, but 2x bigger')

print(f'\n{"─"*60}')
print('EUCLIDEAN DISTANCE (subtract, square, sum, sqrt)')
print(f'{"─"*60}')
dist_ab = _np.sqrt(((a - b)**2).sum())
dist_ac = _np.sqrt(((a - c)**2).sum())
print(f'  A vs B: {dist_ab:.3f}   <- close (good)')
print(f'  A vs C: {dist_ac:.3f}   <- far apart! but C is just A doubled...')

print(f'\n{"─"*60}')
print('COSINE SIMILARITY (multiply pairs, sum, divide by lengths)')
print(f'{"─"*60}')

print(f'\n  A vs B, step by step:')
print(f'  ┌ Step 1: multiply each pair and sum (dot product)')
pairs = '  +  '.join(f'{x}*{y}' for x, y in zip(a, b))
dot_ab = (a * b).sum()
print(f'  │   {pairs}  =  {dot_ab:.3f}')
len_a = _np.linalg.norm(a)
len_b = _np.linalg.norm(b)
print(f'  ├ Step 2: compute length of each vector')
print(f'  │   length A = {len_a:.3f},  length B = {len_b:.3f}')
cos_ab = dot_ab / (len_a * len_b)
print(f'  └ Step 3: divide')
print(f'      cosine = {dot_ab:.3f} / ({len_a:.3f} x {len_b:.3f}) = {cos_ab:.3f}')

len_c = _np.linalg.norm(c)
cos_ac = (a * c).sum() / (len_a * len_c)

print(f'\n{"─"*60}')
print('RESULTS')
print(f'{"─"*60}')
print(f'                    Euclidean    Cosine')
print(f'  A vs B (similar)    {dist_ab:.3f}        {cos_ab:.3f}')
print(f'  A vs C (doubled)    {dist_ac:.3f}        {cos_ac:.3f}')
print(f'\n  Euclidean says A and C are far apart.')
print(f'  Cosine says they are identical - same pattern, different scale.')
print(f'  For recommendations, cosine is right.')


In [ ]:
# Find movies most similar to a given movie
def find_similar(title_substring, top_n=5):
    """Find the top_n most similar movies to the first movie matching title_substring."""
    # Find the movie
    matches = {idx: t for idx, t in idx2title.items() if title_substring.lower() in t.lower()}
    if not matches:
        print(f"No movie found matching '{title_substring}'")
        return
    idx = list(matches.keys())[0]
    title = matches[idx]

    # Compute cosine similarity against all movies
    movie_embs_t = model_v3.movie_factors.weight.detach().cpu()
    target = movie_embs_t[idx].unsqueeze(0)  # (1, 50)
    similarities = F.cosine_similarity(movie_embs_t, target)  # (n_movies,)

    # Sort and skip itself (similarity = 1.0)
    top_idxs = similarities.argsort(descending=True)[1:top_n+1]
    print(f"Movies most similar to '{title}':")
    for i in top_idxs:
        print(f"  {idx2title[i.item()]:45s} similarity: {similarities[i]:.3f}")

find_similar("Silence of the Lambs")
print()
find_similar("Star Wars")

In [ ]:
# "Because you watched..." recommendations
for query in ['Toy Story', 'Godfather', 'Fargo', 'Scream']:
    find_similar(query)
    print()


Some results are strong (Star Wars -> Empire Strikes Back is spot on), others are weaker. 
With only 100K ratings, embeddings for less popular movies are noisier - they haven't had enough gradient updates to settle into the right neighborhood. 
With millions of ratings (like Netflix or Spotify have), these similarity results get much more reliable.

<details>
<summary><strong>Optional experiment: cosine vs Euclidean distance - does it matter?</strong></summary>

Euclidean distance ("how far apart are the numbers") is simpler and also works for finding similar items. 
When does cosine actually make a difference? Let's test it on our trained model.
</details>

In [ ]:
# Cosine vs Euclidean: do they give different recommendations?
movie_embs_all = model_v3.movie_factors.weight.detach().cpu()

def find_similar_euclidean(title_substring, top_n=5):
    """Find similar movies using Euclidean distance instead of cosine."""
    matches = {idx: t for idx, t in idx2title.items() if title_substring.lower() in t.lower()}
    if not matches: return []
    idx = list(matches.keys())[0]
    target = movie_embs_all[idx].unsqueeze(0)
    dists = torch.cdist(movie_embs_all, target).squeeze()
    top = dists.argsort()[1:top_n+1]
    return [idx2title[i.item()] for i in top]

def find_similar_cosine(title_substring, top_n=5):
    """Find similar movies using cosine similarity."""
    matches = {idx: t for idx, t in idx2title.items() if title_substring.lower() in t.lower()}
    if not matches: return []
    idx = list(matches.keys())[0]
    target = movie_embs_all[idx].unsqueeze(0)
    sims = F.cosine_similarity(movie_embs_all, target)
    top = sims.argsort(descending=True)[1:top_n+1]
    return [idx2title[i.item()] for i in top]

# Check how much embedding magnitudes vary
norms = movie_embs_all.norm(dim=1)
print(f'Embedding magnitudes: mean={norms.mean():.2f}  std={norms.std():.2f}  min={norms.min():.2f}  max={norms.max():.2f}')
print(f'Ratio max/min: {norms.max()/norms.min():.1f}x\n')

# Compare results for popular and less popular movies
for query in ['Star Wars', 'Toy Story', 'Silence of the Lambs']:
    cos = find_similar_cosine(query, 3)
    euc = find_similar_euclidean(query, 3)
    overlap = len(set(cos) & set(euc))
    
    print(f'  "{query}":')
    print(f'    Cosine:    {cos[0][:35]}, {cos[1][:35]}, {cos[2][:35]}')
    print(f'    Euclidean: {euc[0][:35]}, {euc[1][:35]}, {euc[2][:35]}')
    print(f'    Overlap: {overlap}/3 same movies\n')


For popular movies like Star Wars (whose embeddings are well-trained from many ratings), cosine and Euclidean usually agree. 
For less popular movies, they can diverge - Euclidean gets distracted by magnitude differences between heavily-rated and rarely-rated movies.

Weight decay helps by keeping embedding magnitudes more uniform, which makes the two metrics more similar. 
But cosine is the safer default because it handles scale differences automatically.

### Global signals: what's universally loved?

The movie bias captures something different from taste match. 
It answers: "regardless of who's watching, do people tend to rate this higher or lower than expected?" 
A high bias means the movie is broadly liked even by users whose taste doesn't particularly align with it. 
This is useful as a fallback for cold-start users, for popular defaults, and for editorial curation.

In [ ]:
# Extract movie biases from the trained model
movie_bias = model_v3.movie_bias.weight.detach().cpu().squeeze()  # (n_movies,)

# 5 lowest biases - universally disliked
worst_idxs = movie_bias.argsort()[:5]
print("Lowest movie biases (universally disliked):")
for idx in worst_idxs:
    print(f"  {idx2title[idx.item()]:45s} bias: {movie_bias[idx]:.3f}")

# 5 highest biases - universally loved
best_idxs = movie_bias.argsort(descending=True)[:5]
print("\nHighest movie biases (universally loved):")
for idx in best_idxs:
    print(f"  {idx2title[idx.item()]:45s} bias: {movie_bias[idx]:.3f}")

Nobody told the model these movies are "good" or "bad." 
It discovered this from the pattern of ratings. 
Bias is more informative than simple average rating because it controls for taste match - a movie could have a high average just because the right audience found it.

### Where this shows up in real products

The same model architecture powers recommendations across very different domains:

- **Streaming** (Netflix, Spotify, YouTube): next watch, autoplay, homepage carousels, "because you watched"
- **E-commerce** (Amazon, Shopify): "customers who bought this", cross-sell, bundle suggestions
- **Content** (news, social media): article recommendations, feed ranking
- **Jobs/marketplaces**: matching candidates to jobs, sellers to buyers
- **Education**: next lesson, next exercise, recommended resources

In all cases, the training objective is the same (predict interactions), but the product use is ranking and retrieval - not the predicted number itself.

In large systems, collaborative filtering often serves as the **first stage**: it narrows millions of items down to a few hundred candidates. 
A second, more expensive model then reranks those candidates using richer features (images, text, user context). 
The embeddings themselves also become reusable features - a user's taste vector can be fed into completely different downstream models.

**A note on scale.** At this dataset size (100K ratings), the model trains in seconds on CPU. 
In production systems with hundreds of millions of interactions, training moves to GPU or distributed computing - 
not because the model architecture is complex (it's still just embeddings + dot products), but because the data volume is massive.

### Inspecting the embedding space with PCA

Each movie has a 50-number embedding vector. We can't visualize 50 dimensions, but we'd like to see whether similar movies ended up near each other. 
PCA (Principal Component Analysis) lets us compress 50 dimensions down to 2 so we can draw a flat picture.

**What PCA actually does:** It doesn't pick 2 of the 50 dimensions and throw away the rest. 
Instead, it creates 2 *new* axes, each a weighted blend of all 50 originals. 
The first axis (PC1) is the direction through 50-dimensional space where the data is most spread out. 
The second axis (PC2) is the direction of most remaining spread, perpendicular to the first.

So PC1 might end up being something like "0.3 x dim1 + 0.1 x dim2 - 0.4 x dim3 + ..." - a mix that happens to capture the biggest pattern in the data. 
Sometimes this aligns with something interpretable ("classic vs modern films"), sometimes it's just "the direction of maximum variation" with no clean label.

The key point: movies that end up near each other in the PCA plot were already near each other in the full 50-dimensional space. 
PCA didn't discover the clusters - the embeddings already contained them. PCA just makes them visible to human eyes.

PCA comes up a lot in ML beyond recommendation systems. 
It's used for dimensionality reduction (reducing features before training), data visualization (plotting high-dimensional clusters), 
noise reduction (keeping only the directions with real signal), and exploratory analysis ("what are the main axes of variation in my data?"). 
Whenever you have more dimensions than you can look at, PCA is usually the first tool to reach for. 
We'll see it again when we work with text embeddings in later lessons.

In [ ]:
# Grab the top 200 most-rated movies for a cleaner plot
movie_rating_counts = ratings.groupby('movie_idx')['rating'].count()
top_movie_idxs = movie_rating_counts.sort_values(ascending=False).head(200).index.values

# Extract their learned embedding vectors and run PCA
movie_embs_np = model_v3.movie_factors.weight.detach().cpu().numpy()
top_embs = movie_embs_np[top_movie_idxs]
pca = PCA(n_components=2)
coords = pca.fit_transform(top_embs)

# Label the top 50 for readability
labels = [idx2title[idx] for idx in top_movie_idxs[:50]]
plot_pca_embeddings(coords, labels, n_show=50, var_ratios=pca.explained_variance_ratio_)


The plot shows recognizable structure - some movies cluster in ways that suggest genre or era patterns. 
The groupings aren't perfectly clean (this is a noisy 2D projection of 50 dimensions from 100K ratings), 
but the fact that any structure emerged at all from pure rating patterns is the point. 
Nobody told the model what action movies or romantic comedies are.

So far we've used a simple dot-product scorer. 
Next we'll ask whether richer models and metadata can improve these rankings, especially when history is sparse.

## Part 7: Beyond Dot Products

The dot product model is elegant but limited in two ways:

1. **Linear only.** The dot product captures "likes action AND likes sci-fi = high score." 
But it can't capture nonlinear interactions like "likes action and comedy separately, but hates action-comedies."
2. **No metadata.** We know things about movies (genres, release year) and users (age, occupation) that the model ignores entirely. 
It learns everything from scratch, even things that are already known.

We'll address both: first with a neural network on top of embeddings, then with a hybrid model that combines embeddings with real features.

### CollabNN: a fair fight

The simplest upgrade: instead of dot-producting the user and movie embeddings, concatenate them into one vector and feed it through an MLP. 
But to make this a fair comparison against DotProductBias, we keep the biases and add dropout.

In [ ]:
class CollabNN(nn.Module):
    def __init__(self, n_users, n_movies, emb_dim=50, hidden=100, y_range=(0, 5.5), p=0.2):
        super().__init__()
        self.user_factors = nn.Embedding(n_users, emb_dim)
        self.movie_factors = nn.Embedding(n_movies, emb_dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.movie_bias = nn.Embedding(n_movies, 1)
        self.layers = nn.Sequential(
            nn.Dropout(p),
            nn.Linear(emb_dim * 2, hidden),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden, 1)
        )
        self.y_range = y_range

        for emb in [self.user_factors, self.movie_factors]:
            nn.init.normal_(emb.weight, 0, 0.1)
        for emb in [self.user_bias, self.movie_bias]:
            nn.init.zeros_(emb.weight)

    def forward(self, users, movies):
        user_emb = self.user_factors(users)
        movie_emb = self.movie_factors(movies)
        x = torch.cat([user_emb, movie_emb], dim=1) # <---- we're concatenating both embedding vectors into our input, similar to how a standard tabular neural network input in the first layer would look like - but we only have embeddings right now. 
        x = self.layers(x).squeeze()
        x += self.user_bias(users).squeeze() + self.movie_bias(movies).squeeze()
        return sigmoid_range(x, *self.y_range)

model_nn = CollabNN(n_users, n_movies)
tl_nn, vl_nn = train_model_es(model_nn, train_dl, valid_dl, n_epochs=30, lr=1e-3, wd=1e-4, patience=5)


Key differences from DotProductBias:
- Embeddings are *concatenated* instead of multiplied - the MLP decides how to combine them
- The ReLU introduces nonlinearity - the model can learn "likes A and B separately but not together"
- Dropout helps regularize the hidden layer
- Biases are added back so the model keeps the strong baseline signals

But we're still only using user ID and movie ID. What about everything else we know?

### Hybrid model: embeddings + metadata

Recommendation systems historically split into two camps:

**Collaborative filtering** (what we've built so far) learns entirely from interaction patterns - who rated what. 
It discovers hidden structure no metadata could capture ("people who like X tend to like Y"), but it needs history. 
New users and new items with no interactions are invisible to it.

**Content-based filtering** uses item and user features directly - genres, cast, age, demographics. 
It can help with cold start (new items have features even without ratings), but it can only recommend items similar to what it already knows about. 
It can't discover that documentary fans also tend to love dark comedies, because that pattern isn't in the metadata.

**Hybrid models** combine both. The idea, first formalized in systems like Microsoft's Deep Crossing (2016) and Google's Wide & Deep (2016), 
is simple: feed learned embeddings AND known features into the same neural network.

In [ ]:
# Hybrid model architecture: embeddings + metadata -> MLP -> rating
plot_hybrid_architecture()


The embeddings (blue, orange) are learned by SGD - they start random and get shaped by training. 
The metadata features (gray) are fixed values from the dataset - age, occupation, genres, year.

Everything gets concatenated into one input vector and fed through the same MLP. 
The network learns how to weight the collaborative signal against the known metadata.

Our MovieLens dataset has exactly the features we need: `u.user` has age and occupation, `u.item` has genres and release year.

In [ ]:
# Load user features
user_features = pd.read_csv(
    DATA_DIR / 'u.user', delimiter='|', header=None,
    names=['user', 'age', 'sex', 'occupation', 'zip']
)

# Load movie features (genres + release date)
genre_names = ['unknown','Action','Adventure','Animation',"Children's",'Comedy','Crime',
               'Documentary','Drama','Fantasy','Film-Noir','Horror','Musical','Mystery',
               'Romance','Sci-Fi','Thriller','War','Western']
item_cols = ['movie','title','release_date','video_release','url'] + genre_names
item_features = pd.read_csv(
    DATA_DIR / 'u.item', delimiter='|', header=None,
    names=item_cols, encoding='latin-1'
)

# Extract release year
item_features['year'] = pd.to_datetime(item_features['release_date'], errors='coerce').dt.year
item_features['year'] = item_features['year'].fillna(item_features['year'].median())

# Occupation -> integer index for embedding
occ_list = sorted(user_features['occupation'].unique())
occ2idx = {o: i for i, o in enumerate(occ_list)}
n_occupations = len(occ_list)

print(f'User features: age (7-73), occupation ({n_occupations} categories)')
print(f'Movie features: {len(genre_names)} genre flags, release year ({int(item_features["year"].min())}-{int(item_features["year"].max())})')


In [ ]:
# Build feature lookup arrays indexed by our contiguous IDs
# User features
user_ages = np.zeros(n_users, dtype=np.float32)
user_occs = np.zeros(n_users, dtype=np.int64)
for uid, uidx in user2idx.items():
    row = user_features[user_features['user'] == uid].iloc[0]
    user_ages[uidx] = row['age']
    user_occs[uidx] = occ2idx[row['occupation']]

# Normalize age to 0-1 range
user_ages = (user_ages - user_ages.min()) / (user_ages.max() - user_ages.min())

# Movie features
movie_genres = np.zeros((n_movies, len(genre_names)), dtype=np.float32)
movie_years = np.zeros(n_movies, dtype=np.float32)
for mid, midx in movie2idx.items():
    row = item_features[item_features['movie'] == mid]
    if len(row) > 0:
        movie_genres[midx] = row[genre_names].values[0]
        movie_years[midx] = row['year'].values[0]

# Normalize year
movie_years = (movie_years - movie_years.min()) / (movie_years.max() - movie_years.min() + 1e-8)

# Convert to tensors
user_ages_t = torch.tensor(user_ages)
user_occs_t = torch.tensor(user_occs)
movie_genres_t = torch.tensor(movie_genres)
movie_years_t = torch.tensor(movie_years)

print(f'Feature shapes: ages={user_ages_t.shape}, occs={user_occs_t.shape}, genres={movie_genres_t.shape}, years={movie_years_t.shape}')


In [ ]:
class HybridModel(nn.Module):
    def __init__(self, n_users, n_movies, n_occs, n_genres, emb_dim=50, occ_dim=8, hidden=100, y_range=(0, 5.5), p=0.2):
        super().__init__()
        # Learned embeddings (collaborative signal)
        self.user_factors = nn.Embedding(n_users, emb_dim)
        self.movie_factors = nn.Embedding(n_movies, emb_dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.movie_bias = nn.Embedding(n_movies, 1)
        self.occ_emb = nn.Embedding(n_occs, occ_dim)
        
        # MLP input: user_emb + movie_emb + age + occ_emb + genres + year
        mlp_input = emb_dim * 2 + 1 + occ_dim + n_genres + 1
        self.layers = nn.Sequential(
            nn.Dropout(p),
            nn.Linear(mlp_input, hidden),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden, 1)
        )
        self.y_range = y_range

        for emb in [self.user_factors, self.movie_factors, self.occ_emb]:
            nn.init.normal_(emb.weight, 0, 0.1)
        for emb in [self.user_bias, self.movie_bias]:
            nn.init.zeros_(emb.weight)

    def forward(self, users, mvs, ages, occs, genres, years):
        user_emb = self.user_factors(users)
        movie_emb = self.movie_factors(mvs)
        occ_emb = self.occ_emb(occs)
        
        # Concatenate everything: embeddings + metadata
        x = torch.cat([
            user_emb, movie_emb,           # learned latent factors
            ages.unsqueeze(1),              # user age (continuous)
            occ_emb,                        # occupation embedding
            genres,                         # genre flags (multi-hot)
            years.unsqueeze(1),             # release year (continuous)
        ], dim=1)
        
        x = self.layers(x).squeeze()
        x += self.user_bias(users).squeeze() + self.movie_bias(mvs).squeeze()
        return sigmoid_range(x, *self.y_range)

print(f'MLP input size: {50*2 + 1 + 8 + 19 + 1} = user_emb(50) + movie_emb(50) + age(1) + occ_emb(8) + genres(19) + year(1)')


The hybrid model takes the same kind of user/movie embeddings as CollabNN (trained from scratch, not reused), but also receives:
- **Age** (continuous, normalized to 0-1)
- **Occupation** (categorical, through its own small embedding)
- **Genre flags** (19 binary values - a movie can be both Action and Sci-Fi)
- **Release year** (continuous, normalized)

Everything gets concatenated into one vector and fed through the MLP. 
The model learns how to weight the collaborative signal (embeddings) against the known metadata.

In [ ]:
# Training function for the hybrid model (needs extra feature inputs)
def train_hybrid(model, train_dl, valid_dl, n_epochs=30, lr=1e-3, wd=1e-4, patience=5):
    model = model.to(device)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.MSELoss()
    
    # Move feature tensors to device
    ages_d = user_ages_t.to(device)
    occs_d = user_occs_t.to(device)
    genres_d = movie_genres_t.to(device)
    years_d = movie_years_t.to(device)
    
    train_losses, valid_losses = [], []
    best_val_loss = float('inf')
    best_state = None
    wait = 0
    
    for epoch in range(n_epochs):
        model.train()
        batch_losses = []
        for users, mvs, rats in train_dl:
            users, mvs, rats = users.to(device), mvs.to(device), rats.to(device)
            preds = model(users, mvs, ages_d[users], occs_d[users], genres_d[mvs], years_d[mvs])
            loss = criterion(preds, rats)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
        
        model.eval()
        all_preds, all_actual = [], []
        with torch.no_grad():
            for users, mvs, rats in valid_dl:
                users, mvs, rats = users.to(device), mvs.to(device), rats.to(device)
                preds = model(users, mvs, ages_d[users], occs_d[users], genres_d[mvs], years_d[mvs])
                all_preds.append(preds.cpu())
                all_actual.append(rats.cpu())
        
        all_preds = torch.cat(all_preds)
        all_actual = torch.cat(all_actual)
        tl = np.mean(batch_losses)
        vl = ((all_preds - all_actual) ** 2).mean().item()
        rmse = ((all_preds - all_actual) ** 2).mean().sqrt().item()
        within_1 = ((all_preds - all_actual).abs() <= 1.0).float().mean().item()
        train_losses.append(tl)
        valid_losses.append(vl)
        
        marker = ''
        if vl < best_val_loss:
            best_val_loss = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
            marker = ' *'
        else:
            wait += 1
        
        print(f'Epoch {epoch+1:2d}/{n_epochs}  MSE={tl:.4f}  val_MSE={vl:.4f}  RMSE={rmse:.2f}  within_1={within_1:.0%}{marker}')
        if wait >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break
    
    model.load_state_dict(best_state)
    return train_losses, valid_losses

model_hybrid = HybridModel(n_users, n_movies, n_occupations, len(genre_names))
tl_hybrid, vl_hybrid = train_hybrid(model_hybrid, train_dl, valid_dl)


### Comparing all approaches

In [ ]:
# Final model comparison
import math

all_models = [
    ('DotProductBias (wd)', valid_losses_v3),
    ('CollabNN (biases+dropout)', vl_nn),
    ('Hybrid (embeddings+metadata)', vl_hybrid),
]

print(f'{"Model":<35s}  {"Best MSE":>8s}  {"RMSE":>6s}')
print('-' * 55)
for name, vl in all_models:
    best = min(vl)
    print(f'{name:<35s}  {best:>8.4f}  {math.sqrt(best):>6.2f}')

# Plot comparison
plot_model_comparison(
    [(train_losses_v3, valid_losses_v3), (tl_nn, vl_nn), (tl_hybrid, vl_hybrid)],
    ['DotProductBias', 'CollabNN', 'Hybrid']
)


CollabNN and DotProductBias perform similarly here - the MLP doesn't clearly beat the dot product. 
That's a useful lesson: **richer models are not automatically better on small datasets.** 
The dot product has a strong inductive bias (it directly measures user-movie alignment), and with 100K ratings the MLP doesn't have enough data to discover patterns the dot product can't.

The hybrid model does show improvement - metadata gives it extra signal that pure interaction data can't provide.

### Where metadata helps: sparse users

Overall RMSE might not change much - all models do well on active users with lots of history. 
The real test: how do they perform on users who have rated very few movies? 
That's where metadata should help most, because there's less collaborative signal to learn from.

In [ ]:
# Compare performance on sparse vs dense users AND movies
user_rating_counts = ratings.groupby('user_idx')['rating'].count()
movie_rating_counts = ratings.groupby('movie_idx')['rating'].count()

sparse_users = set(user_rating_counts[user_rating_counts < 30].index.tolist())
dense_users = set(user_rating_counts[user_rating_counts > 100].index.tolist())
sparse_movies = set(movie_rating_counts[movie_rating_counts < 10].index.tolist())
dense_movies = set(movie_rating_counts[movie_rating_counts > 50].index.tolist())

ages_d = user_ages_t.to(device)
occs_d = user_occs_t.to(device)
genres_d = movie_genres_t.to(device)
years_d = movie_years_t.to(device)

def eval_on_subset(model, dl, user_set=None, movie_set=None, is_hybrid=False):
    model.eval()
    errors = []
    with torch.no_grad():
        for users, mvs, rats in dl:
            if user_set is not None:
                mask = torch.tensor([u.item() in user_set for u in users])
            elif movie_set is not None:
                mask = torch.tensor([m.item() in movie_set for m in mvs])
            else:
                mask = torch.ones(len(users), dtype=torch.bool)
            if mask.sum() == 0:
                continue
            users_m, mvs_m, rats_m = users[mask].to(device), mvs[mask].to(device), rats[mask].to(device)
            if is_hybrid:
                preds = model(users_m, mvs_m, ages_d[users_m], occs_d[users_m], genres_d[mvs_m], years_d[mvs_m])
            else:
                preds = model(users_m, mvs_m)
            errors.append((preds.cpu() - rats_m.cpu()).abs())
    return torch.cat(errors).mean().item() if errors else float('nan')

# Users: sparse vs dense
print('USERS (MAE by rating count)')
print(f'{"":<25s}  {"Sparse (<30)":>12s}  {"Dense (>100)":>12s}  {"Gap":>8s}')
print('-' * 62)
for name, model, hybrid in [
    ('DotProductBias', model_v3, False),
    ('CollabNN', model_nn, False),
    ('Hybrid', model_hybrid, True)]:
    su = eval_on_subset(model, valid_dl, user_set=sparse_users, is_hybrid=hybrid)
    du = eval_on_subset(model, valid_dl, user_set=dense_users, is_hybrid=hybrid)
    print(f'{name:<25s}  {su:>12.3f}  {du:>12.3f}  {su-du:>+8.3f}')

print()

# Movies: sparse vs dense
print('MOVIES (MAE by rating count)')
print(f'{"":<25s}  {"Sparse (<10)":>12s}  {"Dense (>50)":>12s}  {"Gap":>8s}')
print('-' * 62)
for name, model, hybrid in [
    ('DotProductBias', model_v3, False),
    ('CollabNN', model_nn, False),
    ('Hybrid', model_hybrid, True)]:
    sm = eval_on_subset(model, valid_dl, movie_set=sparse_movies, is_hybrid=hybrid)
    dm = eval_on_subset(model, valid_dl, movie_set=dense_movies, is_hybrid=hybrid)
    print(f'{name:<25s}  {sm:>12.3f}  {dm:>12.3f}  {sm-dm:>+8.3f}')


Note: Due to this notebook not using hardcoded seeds (e.g we're not trying to remove randomness), your results may vary from run to run, and because we're dealing with such small 
overall changes at times, conclusions aren't 100% waterproof - but generally speaking, here's what I think makes sense:

For **sparse users**, the hybrid model performs best in absolute MAE,
which suggests that metadata does add useful signal when user history is
limited. But the sparse-vs-dense gap is still larger for the richer
models than for `DotProductBias`, so metadata helps without fully
removing the effect of sparsity.

For **sparse movies**, the picture is more mixed. The hybrid model is
strongest on dense movies, but it does not clearly beat `CollabNN` on
sparse movies. That suggests our movie metadata (`genres` and `year`) is
too coarse to provide a strong item-side cold-start advantage by itself.

The broader lesson is that **richer models can improve overall accuracy,
but they still depend heavily on data density**. Simpler models like
`DotProductBias` may be less accurate overall, yet more stable when data
is thin. With richer item features such as cast, plot summaries, or text
embeddings, the hybrid advantage would likely become clearer.

### The Big Reveal: Collaborative Filtering as Tabular Learning

Look at what the Hybrid model actually does: it takes categorical inputs (user ID, movie ID, occupation), 
converts them to embeddings, adds continuous features (age, year, genres), concatenates everything, 
and feeds it through a linear-ReLU-linear network.

That's essentially a tabular model. 
Collaborative filtering isn't a fundamentally different technique - it's tabular ML where the most important columns happen to be high-cardinality categoricals (user ID, movie ID) that need embeddings.

Production systems at Netflix, Spotify, and YouTube follow a similar pattern: learned embeddings for IDs combined with fixed features for metadata, fed through neural networks trained end to end. 
The details vary (attention layers, sequence models, multi-task objectives), but the core idea - learn embeddings from behavior, combine with metadata, predict - is what we built here.

And it's not limited to users and movies. 
Any pair of entities with interaction data - users and products, words and contexts, genes and diseases - can be modeled this way. 
The "collaborative" part means predictions come from collective behavioral patterns, not from features of the entities themselves. 
That's the defining characteristic of collaborative filtering, regardless of domain.

## The Cold Start Problem

Everything we've built depends on one thing: trained embeddings. A user's embedding captures their taste, a movie's embedding captures its qualities. But what about a brand new user who just signed up? Or a movie that was just added to the catalog?

Their embeddings are random noise. The dot product of random noise with anything is meaningless. Let's see this in action.

In [ ]:
# Grab trained embeddings
user_embs = model_v3.user_factors.weight.detach().cpu()
movie_embs = model_v3.movie_factors.weight.detach().cpu()
movie_biases = model_v3.movie_bias.weight.detach().cpu().squeeze()
user_biases = model_v3.user_bias.weight.detach().cpu().squeeze()

# Existing user with a well-trained embedding
existing_user = user_embs[0]
existing_user_bias = user_biases[0]

# New user: random embedding (what they'd start with)
new_user = torch.randn(n_factors)
new_user_bias = torch.tensor(0.0)

# Average user: mean of all trained embeddings
avg_user = user_embs.mean(dim=0)
avg_user_bias = user_biases.mean()

def show_top5(user_vec, user_bias, label):
    """Show top 5 predicted ratings using the full model (dot product + biases + sigmoid_range)."""
    dots = (movie_embs * user_vec).sum(dim=1)
    scores = sigmoid_range(dots + user_bias + movie_biases, 0, 5.5)
    top5 = scores.argsort(descending=True)[:5]
    print(f"{label}:")
    for idx in top5:
        print(f"  {idx2title[idx.item()]:45s} predicted: {scores[idx]:.1f} stars")
    print()

show_top5(existing_user, existing_user_bias, "Existing user - trained embedding")
show_top5(new_user, new_user_bias, "New user - random embedding (garbage)")
show_top5(avg_user, avg_user_bias, "New user - average embedding (safe default)")


The random embedding produces nonsensical recommendations - arbitrary movies with inflated scores just because the random vectors happened to align. 
The average embedding is a safe fallback: it represents "the typical user" and gravitates toward broadly popular, well-rated films. Not personalized, but at least reasonable.

In practice, companies handle cold start with several strategies:

1. **Show popular items** as defaults - the "Trending Now" row you see before Netflix knows you
2. **Onboarding questions** - "Rate these 5 movies" gives enough signal to build a rough embedding
3. **Content features** - use genre, actors, director to bootstrap until interaction data accumulates. This is where the hybrid deep learning version from Part 7 helps: it can combine learned embeddings with known features.

### Feedback Loops and Representation Bias

There's a subtler problem. If you only recommend popular items to new users, they only interact with popular items, which makes those items appear *even more* popular in the training data. 
This creates a positive feedback loop: popular gets more popular, niche content gets buried.

A small group of very active users can disproportionately shape recommendations for everyone. 
If your most prolific raters happen to love anime, the model's "average user" embedding tilts toward anime - and every new user gets anime recommendations, 
whether they want them or not. Those new users then interact with anime (because that's what they're shown), reinforcing the bias.

Real systems need deliberate **exploration** - occasionally showing unexpected items to break these loops. 
This is an active area of research and a reason why recommendation systems need human oversight, not just better loss functions.

## From Notebook to Production

What we built here is **not** a full Netflix or Spotify recommendation stack. It's the core scoring layer: learn useful user/item representations, score user-item pairs, use similarity, and mix in metadata when pure collaborative filtering is not enough.

Modern systems usually add at least three more layers:

1. **Candidate generation** retrieves a few hundred or thousand plausible items quickly instead of scoring the whole catalog with the most expensive model.
2. **Ranking** uses richer user, item, and context features to order those candidates for a specific request.
3. **Re-ranking** adds diversity, freshness, exploration, and business constraints before anything is shown on the page.

They also usually train on **implicit signals** like watch time, completion, skips, rewatches, and clicks rather than explicit star ratings. So the right expectation is: this notebook teaches the foundation that makes modern recommenders understandable, but real systems still require retrieval, ranking metrics, experimentation, and serving infrastructure on top.

## Summary

We built four recommendation models, each exploring a different idea, all in pure PyTorch:

1. **DotProduct** - two embedding lookups + dot product. The simplest collaborative filtering model. 
Limited because it can only capture taste match, not baseline tendencies.

2. **DotProductBias** - added per-user and per-movie biases to capture "this user rates everything high" 
and "everyone loves Shawshank." Needed weight decay to prevent overfitting.

3. **CollabNN** - concatenated embeddings fed through a neural network with biases and dropout. 
Can learn nonlinear interactions between latent factors.

4. **HybridModel** - combines learned embeddings with known metadata (age, occupation, genres, release year). 
A simplified version of the feature-mixing models that often appear inside production recommendation systems.

### Key takeaways

**Embeddings** are lookup tables wrapped in `nn.Parameter` - matrices of learnable numbers indexed by integer ID. 
They start as random noise and, through training, become rich representations that capture hidden structure.

**SGD doesn't care what it's optimizing.** The same forward-loss-backward-update loop works for regression (L3), 
neural networks (L5), and embedding models (this lesson). Only the architecture changes.

**The trained embeddings are useful beyond predictions.** Cosine similarity powers "because you watched X." 
PCA reveals hidden structure. Biases tell you what's universally loved or hated.

**Collaborative filtering is tabular ML** where the most important columns are high-cardinality categoricals (user ID, movie ID). 
Production systems often combine learned embeddings with metadata features, then wrap that model inside larger retrieval, ranking, and re-ranking pipelines.

**Pure collaborative filtering needs history.** The cold start problem is real - new users and items have no signal. 
Hybrid models help by falling back on metadata, but deliberate exploration is still needed to avoid feedback loops.

## Questionnaire

**Embeddings and latent factors**

1. What is a latent factor? Why is it called "latent"?
1. What is a dot product? Calculate one by hand: `[0.85, 0.9, 0.7]` and `[0.9, 0.95, 0.8]`.
1. Why does the dot product of two aligned vectors produce a high score? What about misaligned vectors?
1. What does `nn.Embedding(943, 50)` create? What shape is its weight matrix? How is it different from a plain tensor?
1. Why do we use `sigmoid_range(x, 0, 5.5)` instead of just clamping to `[1, 5]`?
1. What is the difference between an embedding, a latent factor, and an embedding matrix?

**Model architecture**

1. What are the only learnable parameters in the `DotProduct` model?
1. What is a bias in the context of collaborative filtering? Why can't the dot product alone capture it?
1. A user who rates everything 5 stars and a movie that everyone rates 5 stars - which is a user bias and which is a movie bias?
1. What is weight decay? Write the modified loss formula.
1. Why does penalizing large weights reduce overfitting? Think about the embedding distribution comparison (with vs without weight decay).
1. What is the key architectural difference between `DotProduct` and `CollabNN`? What can `CollabNN` learn that `DotProduct` cannot?
1. Why do user and movie embeddings need the same dimension in `DotProduct` but not in `CollabNN`?

**Training and evaluation**

1. What loss function do we use? Why MSE and not cross-entropy?
1. If the MSE is 0.81, what is the RMSE? What does it mean in terms of star ratings?
1. What does a growing gap between training loss and validation loss tell you?
1. Why did changing the learning rate from 1e-2 to 1e-3 make such a big difference?

**Interpretation and deployment**

1. What do the highest and lowest movie biases tell you? Is movie bias the same as average rating?
1. What is PCA doing when we visualize the 50-dimensional embeddings in 2D? Does PCA discover the clusters, or were they already there?
1. What is cosine similarity? Why do we use it instead of Euclidean distance for finding similar movies?
1. Explain the difference between "top picks for you" and "because you watched X" recommendations.
1. What is the cold start problem? Name two strategies to handle it.
1. What is a feedback loop in recommendation systems? Why is it dangerous?
1. What is the difference between collaborative filtering and content-based filtering? Why do hybrid models combine both?
1. Why does the hybrid model help more for users with few ratings than for active users?
1. Name three real-world domains (beyond movies) where collaborative filtering is used.


### Further Research

1. The hybrid model uses age, occupation, genres, and year. What other features from the MovieLens dataset could you add? Would timestamp (time of day, day of week) help?
1. The current model treats ratings as continuous values (MSE loss). What would change if you treated them as 5 discrete classes and used cross-entropy loss instead?
1. Try replacing AdamW with plain SGD with momentum. How does it affect training dynamics and final validation loss?


<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>